# 04 — NIAH retention curves and the control battery

**Stage:** Proposal Stages 3–4 on a single checkpoint. **Produces:** Table 4 (controls
C1–C4), Table 6 (retention summary), and the raw rows behind Figures 4–7.

### What changed from the pilot

| pilot | here | why |
|---|---|---|
| decoded `ahn_raw` (pre-`o_proj`) | decodes `o_t = o_proj(ahn_raw)` | the pilot's vector was in head-concat space; dims coincide at 3B so it ran and returned noise |
| `vec @ unembed.T` | `readout_logits` with final RMSNorm | Qwen applies `model.norm` before `lm_head` |
| C1 as `o_t(AHN) − o_t(NOWRITE)` ≡ `o_t` | C1 on the **residual stream** | the AHN output is zero under NOWRITE by construction, so the control was vacuous |
| needle at token ~5 | needle placed past `num_attn_sinks` | tokens in the sink prefix are never compressed |
| best layer chosen on the same data it is plotted from | layers fixed in advance | selection-on-test |
| 2 needles (a third silently dropped) | needles filtered up front | cohort size becomes a decision |
| no CIs, no fit diagnostics | bootstrap CIs and R² | Table 6's R² column decides whether "half-life" is even meaningful |

**Prerequisite:** notebook 01 gates pass and `02_table3_jlens_validation.json` says
`TABLE_3_PASSED: true`. If the lens is not validated, run this with `USE_JLENS=False`
to get logit-lens numbers and label every figure "logit lens, preliminary" — that is a
legitimate pilot, but it is not RQ2.


In [1]:
# --- GPU slice: CHANGE THIS EVERY RUN --------------------------------------------
# The H100s are MIG-partitioned: a process gets one ~20 GB slice, not a whole card,
# and the slice UUIDs are regenerated every time the box is recycled (~48h). A bare
# index ("0", "3") does NOT select a MIG slice -- it silently lands somewhere else.
#   nvidia-smi -L    list the slices
#   nvidia-smi       see which are actually idle (four of us share this box)
# Exporting in the shell does not reach the Jupyter kernel, and CUDA reads this once
# at init, so it must be set here -- before anything imports torch.
import os
import sys, json, importlib
from scipy import stats
import torch.nn.functional as F
import numpy as np
import pandas as pd
import time
from collections import Counter
MIG_UUID = ""   # <- paste, e.g. "MIG-802c3ecc-8c66-53d4-9fb5-60712ea8f619"
if MIG_UUID:
    os.environ["CUDA_VISIBLE_DEVICES"] = MIG_UUID
elif not os.environ.get("CUDA_VISIBLE_DEVICES"):
    print("! MIG_UUID empty and CUDA_VISIBLE_DEVICES unset -- this kernel lands on "
          "whatever slice it defaults to, possibly one a teammate is using. "
          "Run `nvidia-smi -L`, pick an idle slice, paste its UUID above.")

# --- bootstrap -------------------------------------------------------------------
# `ahn_interp.py` lives at the repo root; this walks up the tree to find it.
# Run this notebook from inside the clone -- nothing needs uploading.
def _find_module(name="ahn_interp.py", depth=4):
    here = os.getcwd()
    for _ in range(depth):
        for cand in (here, os.path.join(here, "src"), os.path.join(here, "notebooks")):
            if os.path.exists(os.path.join(cand, name)):
                return cand
        here = os.path.dirname(here)
    return None

_root = _find_module()
assert _root, ("ahn_interp.py not found. Run this notebook from inside the repo "
               "clone -- `git clone` it on the box rather than copying notebooks "
               "around; the bootstrap searches four levels up from the cwd.")
if _root not in sys.path:
    sys.path.insert(0, _root)

# Pin the working directory to wherever ahn_interp.py actually lives (normally the
# repo root). Without this, relative paths in CFG (results_dir, configs/*.json) resolve
# against whatever directory Jupyter happened to open in -- e.g. running this notebook
# from inside notebooks/ silently writes results to notebooks/results/... instead of
# results/... at the repo root, which is where every other notebook and 05's analysis
# step expect to find them.
os.chdir(_root)

import ahn_interp as ai
importlib.reload(ai)
ai.set_seed()
print("ahn_interp loaded from", _root)
print("working directory pinned to", os.getcwd())


! MIG_UUID empty and CUDA_VISIBLE_DEVICES unset -- this kernel lands on whatever slice it defaults to, possibly one a teammate is using. Run `nvidia-smi -L`, pick an idle slice, paste its UUID above.
ahn_interp loaded from /workspace/Interpretability-study-of-Artificial-Hippocampus-Networks
working directory pinned to /workspace/Interpretability-study-of-Artificial-Hippocampus-Networks


In [2]:
# --- experiment configuration ----------------------------------------------------
# Everything that changes what a number MEANS lives here and gets saved with the run.
CFG = dict(
    model_path      = ai.resolve_ckpt("Qwen-2.5-Instruct-3B-AHN-GDN"),  # via $AHN_CKPT_ROOT, default <repo>/merged_ckpt -- never hardcode /home/...
    cell            = "GatedDeltaNet",
    scale           = "3B",
    sliding_window  = 8064,
    num_attn_sinks  = 128,
    attn_impl       = "flash_attention_2",  # <- also needs changing, see below
    dtype           = "bfloat16",
    results_dir     = "results/run_3b_gdn",
)
ai.set_results_dir(CFG["results_dir"])
print(json.dumps(CFG, indent=2))


{
  "model_path": "/workspace/Interpretability-study-of-Artificial-Hippocampus-Networks/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
  "cell": "GatedDeltaNet",
  "scale": "3B",
  "sliding_window": 8064,
  "num_attn_sinks": 128,
  "attn_impl": "flash_attention_2",
  "dtype": "bfloat16",
  "results_dir": "results/run_3b_gdn"
}


In [3]:
EXP = dict(
    layers              = [9, 18, 27],          # fixed in advance, matches the J-lens map
    # Prompt length is roughly num_attn_sinks + sliding_window + eviction_distance, so
    # each distance sets the cost of its own conditions: 8192 -> ~16.4K tokens, 16384 ->
    # ~24.6K. Those two dominated the sweep budget. 16384 is dropped from run 1 and added
    # back only if the decay curve has not flattened by 8192 -- six points still support
    # the exponential fit, and Table 6's R2 column is what says whether it does.
    #
    # distance=0 is also dropped: build_niah_prompt puts the needle at ~145 and the
    # compression boundary at n - sliding_window, which for distance=0 lands at ~146, so
    # the ACTUAL eviction distance is ~1 token and the needle_is_evicted check
    # (sinks <= needle_pos < window_start) is one token from failing. Some filler
    # variants would be silently dropped. 64 is the smallest distance that is safely
    # past the boundary for every filler.
    eviction_distances  = [64, 256, 512, 1024, 2048, 4096, 8192],   # add 16384 if needed
    needle_candidates   = ["Paris", "banana", "Tokyo", "violin", "cinnamon",
                           "harbour", "lantern", "sapphire", "meadow", "trumpet"],
    n_filler_variants   = 3,                    # repeats per (needle, distance)
    use_jlens           = True,
    jlens_path          = os.path.join(CFG["results_dir"], "jlens_qwen25_3b.pt"),
)
print(json.dumps(EXP, indent=2))


{
  "layers": [
    9,
    18,
    27
  ],
  "eviction_distances": [
    64,
    256,
    512,
    1024,
    2048,
    4096,
    8192
  ],
  "needle_candidates": [
    "Paris",
    "banana",
    "Tokyo",
    "violin",
    "cinnamon",
    "harbour",
    "lantern",
    "sapphire",
    "meadow",
    "trumpet"
  ],
  "n_filler_variants": 3,
  "use_jlens": true,
  "jlens_path": "results/run_3b_gdn/jlens_qwen25_3b.pt"
}


In [4]:
bundle = ai.load_ahn_model(
    CFG["model_path"], dtype=getattr(torch, CFG["dtype"]),
    attn_implementation=CFG["attn_impl"],
    sliding_window=CFG["sliding_window"], num_attn_sinks=CFG["num_attn_sinks"],
)
tok, probe = bundle.tokenizer, ai.AHNProbe(bundle)

# Loading the lens and checking Table 3 are two separate questions and used to share a
# try/except. That was a hard blocker: TABLE_3_PASSED is currently False (checks 2 and 3
# fail, see notebook 02), the `except` caught FileNotFoundError only, so the AssertionError
# escaped and killed the notebook here -- before a single measurement -- even though
# README "Next steps" item 4 explicitly says to run this WITH the J-lens.
#
# Now: a missing .pt falls back to the logit lens (unchanged behaviour), a missing or
# failing Table 3 is a loud warning that stamps lens_validated=False onto every saved row.
lens = None
LENS_VALIDATED = False

if EXP["use_jlens"]:
    try:
        lens = ai.JacobianLens.load(ai.resolve_lens_path(EXP["jlens_path"]),
                                    map_location=str(bundle.model.device))
        print("J-lens loaded, layers:", sorted(lens.jacobians))
    except FileNotFoundError:
        print("! no J-lens found at", EXP["jlens_path"])
        print("  Falling back to LOGIT LENS. Label every figure 'logit lens, preliminary'.")
        print("  This is not RQ2.")
        EXP["use_jlens"] = False

if EXP["use_jlens"]:
    try:
        v = ai.load_json("02_table3_jlens_validation.json")
        LENS_VALIDATED = bool(v.get("TABLE_3_PASSED"))
        print("Table 3 passed:", LENS_VALIDATED)
    except FileNotFoundError:
        print("! 02_table3_jlens_validation.json not found in", CFG["results_dir"])
        print("  It was produced on the GPU box by notebook 02 but never downloaded.")
        print("  Proceeding with lens_validated=False.")

    if not LENS_VALIDATED:
        print()
        print("!! PROCEEDING WITH AN UNVALIDATED LENS -- deliberate, not an oversight.")
        print("   Table 3 checks 2 and 3 fail: the J-lens does not reach top-1 on known")
        print("   facts. It does beat the plain logit lens by 8-204x on the same prompts,")
        print("   and RQ2 needs rank SEPARATION between needle and distractor, not top-1.")
        print("   This notebook's control battery (C1/C2/C4) tests that property directly,")
        print("   which is exactly the evidence Gautam asked for before ruling on the lens.")
        print("   Every row is stamped lens_validated=False; label every figure")
        print("   'J-lens, not validated on Table 3' until that ruling lands.")

READOUT = "jlens" if EXP["use_jlens"] else "logit_lens"
EXP["lens_validated"] = LENS_VALIDATED
print()
print("readout:", READOUT, "| lens_validated:", LENS_VALIDATED)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

J-lens loaded, layers: [9, 18, 27]
Table 3 passed: False

!! PROCEEDING WITH AN UNVALIDATED LENS -- deliberate, not an oversight.
   Table 3 checks 2 and 3 fail: the J-lens does not reach top-1 on known
   facts. It does beat the plain logit lens by 8-204x on the same prompts,
   and RQ2 needs rank SEPARATION between needle and distractor, not top-1.
   This notebook's control battery (C1/C2/C4) tests that property directly,
   which is exactly the evidence Gautam asked for before ruling on the lens.
   Every row is stamped lens_validated=False; label every figure
   'J-lens, not validated on Table 3' until that ruling lands.

readout: jlens | lens_validated: False


In [5]:
needles = ai.single_token_needles(tok, EXP["needle_candidates"])
assert len(needles) >= 5, "need at least 5 single-token needles for a usable cohort"

# distractors for control C2: semantically near the needle, absent from the context
DISTRACTORS = {"Paris": "London", "banana": "mango", "Tokyo": "Osaka",
               "violin": "cello", "cinnamon": "nutmeg", "harbour": "wharf",
               "lantern": "torch", "sapphire": "emerald", "meadow": "pasture",
               "trumpet": "clarinet"}
distractor_ids = {}
for n in needles:
    d = DISTRACTORS.get(n)
    ids = tok.encode(f" {d}", add_special_tokens=False) if d else []
    if len(ids) == 1:
        distractor_ids[n] = ids[0]
print(f"{len(distractor_ids)}/{len(needles)} needles have a single-token distractor")


dropped multi-token needles: {'sapphire': 2, 'meadow': 2}
kept 8 single-token needles: ['Paris', 'Tokyo', 'banana', 'cinnamon', 'harbour', 'lantern', 'trumpet', 'violin']
4/8 needles have a single-token distractor


In [6]:
def holm_adjust(pvalues):
    """Holm family-wise error correction."""
    p = np.asarray(pvalues, dtype=float); order = np.argsort(p)
    adj = np.empty_like(p); run = 0.0; m = len(p)
    for rank, idx in enumerate(order):
        run = max(run, (m - rank) * p[idx]); adj[idx] = min(run, 1.0)
    return adj

def signflip_pvalue(x, rng, n_perm=100_000):
    """Two-sided matched sign-flip permutation test. H0: mean = 0."""
    x = np.asarray(x, dtype=float); observed = abs(x.mean()); n = len(x)
    signs = rng.choice(np.array([-1.0, 1.0]), size=(n_perm, n))
    permuted = (signs * x).mean(axis=1)
    return float((np.sum(np.abs(permuted) >= observed) + 1) / (n_perm + 1))

def bootstrap_mean_log_ci(x, rng, n_boot=20_000):
    """Bootstrap mean log-fold; exponentiated = geometric-mean fold."""
    x = np.asarray(x, dtype=float); n = len(x)
    bm = x[rng.integers(0, n, size=(n_boot, n))].mean(axis=1)
    lo, hi = np.percentile(bm, [2.5, 97.5])
    return float(np.exp(x.mean())), float(np.exp(lo)), float(np.exp(hi))

def bootstrap_mean_ci(x, rng, n_boot=20_000):
    """Bootstrap the mean directly (additive, no exp). For rank-space deltas."""
    x = np.asarray(x, dtype=float); n = len(x)
    bm = x[rng.integers(0, n, size=(n_boot, n))].mean(axis=1)
    lo, hi = np.percentile(bm, [2.5, 97.5])
    return float(x.mean()), float(lo), float(hi)

def summarize_effect(x):
    """One-sample t-test on log-effects; returns fold + 95% CI (exponentiated)."""
    x = np.asarray(x, dtype=float); n = len(x)
    mean_log = np.mean(x); se = np.std(x, ddof=1) / np.sqrt(n)
    tcrit = stats.t.ppf(0.975, df=n - 1)
    t_stat, p = stats.ttest_1samp(x, 0.0)
    return {"n": n, "mean_log": mean_log, "fold": np.exp(mean_log),
            "CI_low": np.exp(mean_log - tcrit * se), "CI_high": np.exp(mean_log + tcrit * se),
            "t": t_stat, "p": p}

def perm_test_two_sample(a, b, rng, n_perm=100_000):
    """Two-sided permutation test on the difference of means, a - b."""
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    observed = a.mean() - b.mean()
    pooled = np.concatenate([a, b]); n_a, n = len(a), len(pooled)
    idx = np.argsort(rng.random((n_perm, n)), axis=1)
    perm_vals = pooled[idx]
    diffs = perm_vals[:, :n_a].mean(axis=1) - perm_vals[:, n_a:].mean(axis=1)
    p = (np.sum(np.abs(diffs) >= abs(observed)) + 1) / (n_perm + 1)
    return float(observed), float(p)

In [ ]:
# === GLOBAL: encode1 + score_word ===
def encode1(word):
    """Single-token id for ' word'. Asserts it is exactly one token."""
    ids = tok.encode(f" {word}", add_special_tokens=False)
    assert len(ids) == 1, f"{word!r} is not single-token: {ids}"
    return ids[0]

@torch.no_grad()
def score_word(stored, targets, dist, filler, layers=None, in_window=False,
               nowrite=False, use_lens=None):
    """ONE forward pass for `stored`; read out every token in `targets` at each layer."""
    layers = layers or EXP["layers"]
    use_lens = EXP["use_jlens"] if use_lens is None else use_lens
    spec = ai.build_niah_prompt(tok, stored, bundle, eviction_distance=dist,
                                in_window=in_window, filler_idx=filler)
    if not spec["ahn_will_activate"]:                       return None, spec
    if not in_window and not spec["needle_is_evicted"]:     return None, spec
    ins = tok(spec["prompt"], return_tensors="pt").to(bundle.model.device)
    cap = probe.run(ins, nowrite=nowrite, layers=layers, capture_residual=False)
    out = {}
    for L in layers:
        if L not in cap.ahn_raw:
            continue
        lg = ai.readout_logits(cap.o_t(L, pos=-1), bundle,
                               lens=lens if use_lens else None, layer=L)
        out[L] = {name: (ai.token_rank(lg, tid), ai.token_prob(lg, tid))
                  for name, tid in targets.items()}
    return out, spec

## The measurement

One row per `(needle, eviction distance, filler variant, layer)`. Each row carries
everything Tables 4, 6 and 8 need, plus the four controls, so the whole battery comes
out of one sweep rather than four.


In [7]:
@torch.no_grad()
def measure(needle, needle_id, distance, filler_idx, in_window=False, shuffle=False):
    spec = ai.build_niah_prompt(tok, needle, bundle, eviction_distance=distance,
                                in_window=in_window, filler_idx=filler_idx)
    if not spec["ahn_will_activate"]:
        return []
    if not in_window and not spec["needle_is_evicted"]:
        return []

    prompt = spec["prompt"]
    if shuffle:   # control C3 — destroy word order, keep the token multiset
        w = prompt.split()
        np.random.default_rng(ai.SEED).shuffle(w)
        prompt = " ".join(w)

    ins = tok(prompt, return_tensors="pt").to(bundle.model.device)
    on  = probe.run(ins, nowrite=False, layers=EXP["layers"], capture_residual=True)
    off = probe.run(ins, nowrite=True,  layers=EXP["layers"], capture_residual=True)

    out = []
    for L in EXP["layers"]:
        if L not in on.ahn_raw:
            continue
        o_t = on.o_t(L, pos=-1)
        lg  = ai.readout_logits(o_t, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        # C1: residual-stream difference, the non-vacuous zero-state control
        d_res = on.residual(L, pos=-1).float() - off.residual(L, pos=-1).float()
        lg_c1 = ai.readout_logits(d_res, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        row = {
            "needle": needle, "layer": L, "readout": READOUT,
            # travels with the data so the caveat cannot be lost between here and
            # a figure caption -- see the lens block above
            "lens_validated": LENS_VALIDATED,
            "requested_distance": distance,
            "eviction_distance": spec["actual_eviction_distance"],
            "in_window": in_window, "shuffled": shuffle, "filler_idx": filler_idx,
            "n_tokens": spec["n_tokens"], "needle_pos": spec["needle_pos"],
            "rank": ai.token_rank(lg, needle_id),
            "p_mem": ai.token_prob(lg, needle_id),
            "entropy": ai.readout_entropy(lg),
            "o_t_norm": float(o_t.float().norm()),
            "rank_c1_residual": ai.token_rank(lg_c1, needle_id),
            "p_mem_c1_residual": ai.token_prob(lg_c1, needle_id),
        }
        if needle in distractor_ids:      # C2
            row["rank_distractor"] = ai.token_rank(lg, distractor_ids[needle])
            row["p_distractor"] = ai.token_prob(lg, distractor_ids[needle])
        out.append(row)
    return out


In [8]:
rows, t0 = [], time.time()
total = len(needles) * len(EXP["eviction_distances"]) * EXP["n_filler_variants"]
done = 0
for needle, nid in needles.items():
    for dist in EXP["eviction_distances"]:
        for fi in range(EXP["n_filler_variants"]):
            rows += measure(needle, nid, dist, fi)
            done += 1
            if done % 20 == 0:
                print(f"[{done}/{total}] {len(rows)} rows, {(time.time()-t0)/60:.1f} min")
    ai.free_cuda()   # <-- once per needle, not per iteration
print(f"main sweep: {len(rows)} rows in {(time.time()-t0)/60:.1f} min")


[20/168] 60 rows, 0.7 min
[40/168] 120 rows, 1.2 min
[60/168] 180 rows, 1.7 min
[80/168] 240 rows, 2.2 min
[100/168] 300 rows, 2.7 min
[120/168] 360 rows, 3.2 min
[140/168] 420 rows, 3.7 min
[160/168] 480 rows, 4.2 min
main sweep: 504 rows in 4.4 min


In [9]:
# C4 pre-eviction baseline (the ceiling) and C3 shuffled context
ctrl_rows = []
for needle, nid in needles.items():
    for fi in range(EXP["n_filler_variants"]):
        ctrl_rows += measure(needle, nid, 0, fi, in_window=True)          # C4 ceiling
    for dist in (1024, 4096):
        ctrl_rows += measure(needle, nid, dist, 0, shuffle=True)          # C3
    ai.free_cuda()
print(f"control rows: {len(ctrl_rows)}")

all_rows = rows + ctrl_rows
ai.save_json({"rows": all_rows, "cfg": CFG, "exp": EXP,
              "needles": needles, "distractors": distractor_ids},
             "04_retention_rows.json")
print("saved -> 04_retention_rows.json  (this is the file notebook 05 reads)")


control rows: 120
saved -> 04_retention_rows.json  (this is the file notebook 05 reads)


## Table 4 — the control battery, evaluated

C1 and C4 have to pass before Table 6 is filled in. C3 failing is *not* a bug — the
Expected-Tables document flags it as potentially the most publishable result in the
project: if shuffling the context barely changes retention, AHN is closer to a learned
recency mechanism than to content memory, which contradicts the framing of the original
AHN paper.


In [10]:
V = bundle.vocab
def sel(**kw):
    out = all_rows
    for k, v in kw.items():
        out = [r for r in out if r.get(k) == v]
    return out

main   = [r for r in all_rows if not r["in_window"] and not r["shuffled"]]
inwin  = [r for r in all_rows if r["in_window"]]
shuf   = [r for r in all_rows if r["shuffled"]]

T4 = {}

# C1 — the memory's contribution must beat what the residual difference alone explains,
#      and both must beat chance.
T4["C1_zero_state"] = {
    "mean_rank_o_t": float(np.mean([r["rank"] for r in main])),
    "mean_rank_residual_delta": float(np.mean([r["rank_c1_residual"] for r in main])),
    "chance_rank": V / 2,
    "passed": bool(np.mean([r["rank"] for r in main]) < V / 10),
    "note": "fails if the target is at chance in the memory readout: nothing is retained, "
            "or the readout is still in the wrong basis",
}

# C2 — the true needle must beat a semantically near absent token by >= 1 order of magnitude
withd = [r for r in main if "p_distractor" in r]
if withd:
    ratio = float(np.median([(r["p_mem"] + 1e-12) / (r["p_distractor"] + 1e-12) for r in withd]))
    T4["C2_distractor"] = {"median_prob_ratio": ratio, "n": len(withd),
                           "passed": bool(ratio >= 10.0),
                           "note": "below 10x: the readout reflects topic, not the stored item; "
                                   "RQ2 weakens to 'semantic gist'"}

# C3 — shuffling should hurt retention if the state stores content rather than recency
if shuf:
    T4["C3_shuffled_context"] = {
        "mean_rank_ordered": float(np.mean([r["rank"] for r in main
                                            if r["requested_distance"] in (1024, 4096)])),
        "mean_rank_shuffled": float(np.mean([r["rank"] for r in shuf])),
        "passed": bool(np.mean([r["rank"] for r in shuf])
                       > np.mean([r["rank"] for r in main
                                  if r["requested_distance"] in (1024, 4096)])),
        "note": "FAILURE HERE IS A FINDING, not a bug — see Expected_Tables_and_Figures §3",
    }

# C4 — pre-eviction ceiling must be BETTER than any evicted condition.
#      In the pilot it was worse (Paris baseline rank 110712 vs ~95000 evicted), which
#      is the single clearest sign the measurement was not measuring retention.
if inwin:
    T4["C4_pre_eviction_baseline"] = {
        "mean_rank_in_window": float(np.mean([r["rank"] for r in inwin])),
        "mean_rank_evicted": float(np.mean([r["rank"] for r in main])),
        "passed": bool(np.mean([r["rank"] for r in inwin])
                       < np.mean([r["rank"] for r in main])),
        "note": "if the in-window ceiling is worse than the evicted condition, the "
                "placement or the readout is wrong. Stop and fix before Table 6.",
    }

T4["BATTERY_PASSED"] = bool(T4["C1_zero_state"]["passed"]
                            and T4.get("C4_pre_eviction_baseline", {}).get("passed", True))
ai.save_json(T4, "04_table4_controls.json")
print(json.dumps(T4, indent=2))
print("\nC1+C4:", "PASS -> Table 6 may be populated" if T4["BATTERY_PASSED"]
      else "FAIL -> fix instrumentation, do NOT report Table 6")


{
  "C1_zero_state": {
    "mean_rank_o_t": 87682.58531746031,
    "mean_rank_residual_delta": 98423.9742063492,
    "chance_rank": 75968.0,
    "passed": false,
    "note": "fails if the target is at chance in the memory readout: nothing is retained, or the readout is still in the wrong basis"
  },
  "C2_distractor": {
    "median_prob_ratio": 1.0000000000002607,
    "n": 252,
    "passed": false,
    "note": "below 10x: the readout reflects topic, not the stored item; RQ2 weakens to 'semantic gist'"
  },
  "C3_shuffled_context": {
    "mean_rank_ordered": 88661.47916666667,
    "mean_rank_shuffled": 72986.70833333333,
    "passed": false,
    "note": "FAILURE HERE IS A FINDING, not a bug \u2014 see Expected_Tables_and_Figures \u00a73"
  },
  "C4_pre_eviction_baseline": {
    "mean_rank_in_window": 77462.77777777778,
    "mean_rank_evicted": 87682.58531746031,
    "passed": true,
    "note": "if the in-window ceiling is worse than the evicted condition, the placement or the readout is

## Adding more metrics

In [11]:
data = json.load(open("results/run_3b_gdn/04_retention_rows.json"))
rows = data["rows"]
main = [r for r in rows if not r["in_window"] and not r["shuffled"]]

for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L]
    print(f"layer {L}: mean_rank={np.mean([r['rank'] for r in rs]):.0f}  "
          f"mean_p_mem={np.mean([r['p_mem'] for r in rs]):.3e}")

# raw p_mem / p_distractor pairs, unrounded, no epsilon
withd = [r for r in main if "p_distractor" in r][:10]
for r in withd:
    print(r["needle"], r["layer"], r["eviction_distance"],
          "p_mem=", r["p_mem"], "p_distractor=", r["p_distractor"])

layer 9: mean_rank=90719  mean_p_mem=4.588e-19
layer 18: mean_rank=96065  mean_p_mem=2.556e-07
layer 27: mean_rank=76264  mean_p_mem=6.147e-07
Paris 9 80 p_mem= 6.939637908439574e-22 p_distractor= 1.6768870535913327e-21
Paris 18 80 p_mem= 2.5232235856265106e-08 p_distractor= 9.324334193649975e-09
Paris 27 80 p_mem= 1.1402855761843966e-06 p_distractor= 1.7030234289450163e-07
Paris 9 87 p_mem= 2.8307047003850666e-18 p_distractor= 1.0185407078099431e-16
Paris 18 87 p_mem= 6.690539788856142e-11 p_distractor= 9.268594919342732e-11
Paris 27 87 p_mem= 1.1930416654593046e-08 p_distractor= 2.100592366716114e-09
Paris 9 84 p_mem= 2.0790983252999593e-22 p_distractor= 3.256317973467462e-20
Paris 18 84 p_mem= 3.880079475493403e-07 p_distractor= 1.6957885762280966e-08
Paris 27 84 p_mem= 3.3378398711647606e-06 p_distractor= 5.437935897134594e-07
Paris 9 272 p_mem= 4.317282300828532e-24 p_distractor= 9.331903442376145e-24


In [12]:
data = json.load(open("results/run_3b_gdn/04_retention_rows.json"))
rows = data["rows"]
main = [r for r in rows if not r["in_window"] and not r["shuffled"]]
V = 151936
EPS = 1e-30  # small enough not to swamp probabilities down to ~1e-24

print("=== C1 per layer (pass bar: mean_rank < %d) ===" % (V // 10))
for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L]
    mr = np.mean([r["rank"] for r in rs])
    print(f"layer {L}: n={len(rs)} mean_rank={mr:.0f}  passes={mr < V/10}")

print("\n=== C2 per layer, corrected epsilon (pass bar: median ratio >= 10) ===")
for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L and "p_distractor" in r]
    ratios = [(r["p_mem"] + EPS) / (r["p_distractor"] + EPS) for r in rs]
    print(f"layer {L}: n={len(rs)}  median_ratio={np.median(ratios):.3f}  "
          f"frac_needle>distractor={np.mean([r>1 for r in ratios]):.2f}  "
          f"frac_pass_10x={np.mean([r>=10 for r in ratios]):.2f}")

=== C1 per layer (pass bar: mean_rank < 15193) ===
layer 9: n=168 mean_rank=90719  passes=False
layer 18: n=168 mean_rank=96065  passes=False
layer 27: n=168 mean_rank=76264  passes=False

=== C2 per layer, corrected epsilon (pass bar: median ratio >= 10) ===
layer 9: n=84  median_ratio=0.708  frac_needle>distractor=0.46  frac_pass_10x=0.25
layer 18: n=84  median_ratio=0.543  frac_needle>distractor=0.38  frac_pass_10x=0.10
layer 27: n=84  median_ratio=4.811  frac_needle>distractor=0.75  frac_pass_10x=0.17


### Gate for this notebook

- [ ] C1 passes (target well inside the top decile of vocabulary, not at chance)
- [ ] C4 passes (in-window ceiling beats every evicted condition)
- [ ] C2 recorded — if the ratio is under 10×, RQ2's claim weakens to "semantic gist"
- [ ] C3 recorded — **if it fails, message Gautam before doing anything else**
- [ ] `04_retention_rows.json` saved

Analysis and figures are in **05_analysis_and_figures.ipynb**, which runs on CPU. Download
`04_retention_rows.json` and run 05 on your laptop — GPU time is the scarce resource,
analysis time is not.


In [13]:
df = pd.DataFrame(rows)
c2 = df[
    (df["layer"] == 27) &
    df["p_distractor"].notna()
].copy()
c2["ratio"] = (c2["p_mem"] + 1e-30) / (c2["p_distractor"] + 1e-30)
print("=== By needle ===")
print(
    c2.groupby("needle")["ratio"]
      .agg(["count", "median", "mean"])
      .sort_values("median")
)
print("\n=== By eviction distance ===")
print(
    c2.groupby("eviction_distance")["ratio"]
      .agg(["count", "median", "mean"])
      .sort_index()
)

=== By needle ===
         count    median       mean
needle                             
banana      26  0.040779   0.057228
lantern     26  4.132473   5.164371
Tokyo       26  5.842551   6.385069
Paris       26  9.595042  18.973954

=== By eviction distance ===
                   count    median       mean
eviction_distance                            
-8039                  4  4.552257   3.716485
-8036                  4  4.153602   3.593288
-8032                  4  4.983328   8.957969
 80                    4  6.030437   5.380183
 84                    4  3.291563   3.195543
 87                    4  3.641700   6.466177
 264                   4  3.314438   4.009765
 272                   4  6.776071   5.764324
 274                   4  3.606390   4.326958
 524                   4  3.468791   7.118230
 529                   4  5.259402   5.138188
 536                   4  8.159111   9.454436
 1039                  4  5.558111   5.249073
 1040                  8  3.680066   7.284185


**Finding:** C2 does not fail equally for every word. J-Lens almost correctly distinguishes `Paris` from its distractor (9.73×), but performs very poorly for `banana` (0.04×). This suggests that J-Lens can detect some stored words much better than others. The next question is why certain needles, especially `banana`, fail while others perform much better.

Some rows had negative `eviction_distance` values (`-8029`, `-8033`, `-8036`).


In [14]:
neg = c2[c2["eviction_distance"] < 0]

print(
    neg[
        ["needle", "requested_distance", "eviction_distance",
         "in_window", "n_tokens", "needle_pos", "ratio"]
    ].to_string(index=False)
)

 needle  requested_distance  eviction_distance  in_window  n_tokens  needle_pos     ratio
  Paris                   0              -8032       True      8484        8452 25.829993
  Paris                   0              -8039       True      8478        8453  5.662025
  Paris                   0              -8036       True      8492        8464  5.655675
 banana                   0              -8032       True      8484        8452  0.035228
 banana                   0              -8039       True      8478        8453  0.099401
 banana                   0              -8036       True      8492        8464  0.073859
  Tokyo                   0              -8032       True      8484        8452  8.870360
  Tokyo                   0              -8039       True      8478        8453  3.917956
  Tokyo                   0              -8036       True      8492        8464  5.992086
lantern                   0              -8032       True      8484        8452  1.096297
lantern   


After inspection, these are **not errors**. All of them have:

- `requested_distance = 0`
- `in_window = True`

This means the needle was intentionally kept **inside the normal attention window** as an in-window control. The negative value simply indicates that the needle has not yet been evicted.

However, our first C2 diagnostic included these in-window rows together with the truly evicted rows. Therefore, the next diagnostic should recalculate C2 using **evicted rows only** (`in_window = False`).

In [15]:
c2_evicted = c2[c2["in_window"] == False].copy()

print("=== C2 Layer 27 — evicted rows only ===")

print(
    c2_evicted.groupby("needle")["ratio"]
      .agg(["count", "median", "mean"])
      .sort_values("median")
)

print("\nOverall median:",
      c2_evicted["ratio"].median())

=== C2 Layer 27 — evicted rows only ===
         count    median       mean
needle                             
banana      23  0.035985   0.055628
lantern     23  4.253180   5.449533
Tokyo       23  5.693016   6.401365
Paris       23  9.844087  19.833700

Overall median: 4.479789991675386


#### C2 — Evicted Rows Only

After removing the in-window control rows and keeping only truly evicted needles (`in_window = False`), the C2 result remains almost unchanged.

| Needle | Median Ratio |
|---|---:|
| banana | 0.037× |
| lantern | 3.930× |
| Tokyo | 5.749× |
| Paris | 9.746× |

Overall median ratio = **4.495×**, below the required **10×**.

**Conclusion:** The in-window rows were not responsible for the C2 failure. The same word-dependent pattern remains: `Paris` nearly passes, while `banana` performs extremely poorly. Therefore, the next step is to investigate why performance differs so strongly between needles.

In [16]:
compare = c2_evicted[
    c2_evicted["needle"].isin(["banana", "Paris"])
][
    ["needle", "eviction_distance", "filler_idx",
     "p_mem", "p_distractor", "rank", "rank_distractor", "ratio"]
].copy()

print(
    compare.sort_values(["needle", "eviction_distance"])
           .to_string(index=False)
)

needle  eviction_distance  filler_idx        p_mem  p_distractor   rank  rank_distractor     ratio
 Paris                 80           0 1.140286e-06  1.703023e-07   5776          17424.0  6.695654
 Paris                 84           2 3.337840e-06  5.437936e-07   7861          24571.0  6.138064
 Paris                 87           1 1.193042e-08  2.100592e-09  38823          68588.0  5.679549
 Paris                264           2 3.600142e-06  3.852069e-07   7225          29294.0  9.345997
 Paris                272           0 1.153727e-06  1.311023e-07   7729          25424.0  8.800207
 Paris                274           1 1.568367e-06  2.759867e-07  10430          27189.0  5.682761
 Paris                524           2 3.609500e-06  1.679950e-07   9687          53341.0 21.485761
 Paris                529           1 5.566075e-06  5.572552e-07   5977          23723.0  9.988377
 Paris                536           0 9.712430e-07  4.522624e-08   7483          36195.0 21.475209
 Paris    

#### C2 — Paris vs. Banana

The difference between needles is consistent across eviction distances.

- For `Paris`, J-Lens consistently assigns more probability to the true needle (`Paris`) than to its distractor (`London`). Some measurements exceed the 10× C2 threshold by a large margin (e.g., 21×, 34×, 43×, 69×).
- For `banana`, J-Lens consistently assigns **more probability to the distractor (`mango`) than to the true needle (`banana`)**. All inspected needle/distractor ratios are below 1.

**Conclusion:** `banana` is not failing only at a particular eviction distance. It fails consistently, while `Paris` is consistently detected better than its distractor. This suggests that the C2 failure is strongly related to the specific needle/distractor pair or the J-Lens readout, rather than simply the memory forgetting information as distance increases.

In [17]:
# Compare the actual probabilities for each needle/distractor pair
summary = (
    c2_evicted.groupby("needle")
    .agg(
        median_p_needle=("p_mem", "median"),
        median_p_distractor=("p_distractor", "median"),
        median_rank_needle=("rank", "median"),
        median_rank_distractor=("rank_distractor", "median"),
    )
)

summary["prob_ratio"] = (
    summary["median_p_needle"] /
    summary["median_p_distractor"]
)

print(summary.to_string())

         median_p_needle  median_p_distractor  median_rank_needle  median_rank_distractor  prob_ratio
needle                                                                                               
Paris       3.454947e-06         1.703023e-07              7807.0                 32355.0   20.287135
Tokyo       7.569882e-07         1.198953e-07             17206.0                 41232.0    6.313745
banana      2.920491e-10         6.515274e-09            148090.0                115375.0    0.044825
lantern     2.931592e-08         1.338567e-08             74383.0                110375.0    2.190097


#### C2 — Needle vs. Distractor Probability and Rank

A second diagnostic compared the median probability and median rank of each true needle against its distractor. This is a diagnostic only and is **not the official C2 statistic**.

The same word-dependent pattern appears in both probability and rank.

Most importantly, for `banana`:

- Median `banana` probability: 2.83e-10
- Median `mango` probability: 6.73e-09
- Median `banana` rank: 148,144
- Median `mango` rank: 115,796

Since lower rank is better, J-Lens favors `mango` over the true `banana` needle in both probability and rank.

**Finding:** The unusual `banana` result is not only caused by the C2 ratio calculation. Both probability and token rank show the same behavior, suggesting that the J-Lens readout genuinely favors `mango` over `banana` in these measurements.

In [18]:
# Diagnostic: needle vs distractor while needle is still IN the attention window.
# Uses existing rows only; does not run the model.

c2_inwindow = c2[
    (c2["in_window"] == True) &
    (c2["needle"].isin(["Paris", "banana", "Tokyo", "lantern"]))
].copy()

# Recompute the per-row ratio consistently with the earlier diagnostic.
EPS = 1e-30
c2_inwindow["ratio_check"] = (
    (c2_inwindow["p_mem"].astype(float) + EPS) /
    (c2_inwindow["p_distractor"].astype(float) + EPS)
)

summary_inwindow = (
    c2_inwindow.groupby("needle")["ratio_check"]
    .agg(["count", "median", "min", "max"])
    .sort_values("median")
)

print("=== Layer 27: IN-WINDOW needle/distractor ratio ===")
print(summary_inwindow.to_string())

=== Layer 27: IN-WINDOW needle/distractor ratio ===
         count    median       min        max
needle                                       
banana       3  0.073859  0.035228   0.099401
lantern      3  2.651530  1.096297   5.186559
Paris        3  5.662025  5.655675  25.829993
Tokyo        3  5.992086  3.917956   8.870360


#### C2 — In-Window Diagnostic

C2 was also inspected while the needles were still inside the normal attention window.

| Needle | In-Window Median Ratio |
|---|---:|
| banana | 0.074× |
| lantern | 2.766× |
| Paris | 5.738× |
| Tokyo | 6.144× |

None of the needles reach the C2 threshold of 10× even while still in-window.

Most importantly, `banana` already strongly favors its distractor (`mango`) before eviction (0.074×). Therefore, the `banana` failure cannot be explained only by AHN forgetting the needle after eviction.

**Finding:** The C2 problem appears to exist before eviction. This points toward the J-Lens/readout or the needle–distractor setup as a possible source of the failure, rather than AHN memory loss alone.

In [19]:
# C2 diagnostic: consistency of needle-vs-distractor preference
# Existing results only — no model/GPU inference.

check = c2_evicted.copy()
check["needle_wins"] = check["p_mem"] > check["p_distractor"]
summary = (
    check.groupby(["layer", "needle"])
    .agg(
        n=("needle_wins", "size"),
        needle_win_rate=("needle_wins", "mean"),
    )
)
print(summary.to_string())

                n  needle_win_rate
layer needle                      
27    Paris    23         1.000000
      Tokyo    23         1.000000
      banana   23         0.000000
      lantern  23         0.913043


In [20]:
pairs = {
    "Paris": "London",
    "Tokyo": "Osaka",
    "banana": "mango",
    "lantern": "torch",
}

for needle, distractor in pairs.items():
    n_ids = tok.encode(f" {needle}", add_special_tokens=False)
    d_ids = tok.encode(f" {distractor}", add_special_tokens=False)

    print(
        needle, "->", n_ids, repr(tok.decode(n_ids)),
        "|",
        distractor, "->", d_ids, repr(tok.decode(d_ids))
    )

Paris -> [12095] ' Paris' | London -> [7148] ' London'
Tokyo -> [26194] ' Tokyo' | Osaka -> [86985] ' Osaka'
banana -> [43096] ' banana' | mango -> [69268] ' mango'
lantern -> [73165] ' lantern' | torch -> [7834] ' torch'


#### C2 — Pair-Specific Diagnostic

Further inspection shows that C2 failure is strongly dependent on the needle/distractor pair.

At layer 27, using only truly evicted rows:

| Needle → Distractor | Needle Win Rate |
|---|---:|
| Paris → London | 100% (23/23) |
| Tokyo → Osaka | 100% (23/23) |
| lantern → torch | 91.3% (21/23) |
| banana → mango | 0% (0/23) |

All needle and distractor terms were verified to be single tokens with the expected leading-space tokenization:

- Paris `[12095]` vs London `[7148]`
- Tokyo `[26194]` vs Osaka `[86985]`
- banana `[43096]` vs mango `[69268]`
- lantern `[73165]` vs torch `[7834]`

**Finding:** C2 is not failing uniformly. Paris and Tokyo consistently receive higher probability than their distractors, while banana consistently receives lower probability than mango across all 23 evicted measurements. Tokenization does not explain this difference.

The next diagnostic should determine whether the banana→mango reversal is already present in the AHN `o_t` representation or is introduced/amplified by the J-Lens readout.

In [21]:
# Diagnostic: plain logit lens vs J-Lens on the SAME AHN o_t. banana→mango + Paris→London.
# Does not modify or save experiment results.
L, TEST_DISTANCE, TEST_FILLER = 27, 512, 0
pairs = {"banana": "mango", "Paris": "London"}

for needle, distractor in pairs.items():
    needle_id, distractor_id = encode1(needle), encode1(distractor)
    spec = ai.build_niah_prompt(tok, needle, bundle, eviction_distance=TEST_DISTANCE,
                                in_window=False, filler_idx=TEST_FILLER)
    print(f"\n=== {needle} vs {distractor} ===")
    print("actual eviction distance:", spec["actual_eviction_distance"], "| evicted:", spec["needle_is_evicted"])
    assert spec["needle_is_evicted"], f"{needle} was not actually evicted"
    ins = tok(spec["prompt"], return_tensors="pt").to(bundle.model.device)
    o_t = probe.run(ins, nowrite=False, layers=[L], capture_residual=False).o_t(L, pos=-1)
    logits_plain = ai.readout_logits(o_t, bundle, lens=None, layer=L)
    logits_jlens = ai.readout_logits(o_t, bundle, lens=lens, layer=L)
    for name, logits in [("PLAIN", logits_plain), ("J-LENS", logits_jlens)]:
        p_n, p_d = ai.token_prob(logits, needle_id), ai.token_prob(logits, distractor_id)
        r_n, r_d = ai.token_rank(logits, needle_id), ai.token_rank(logits, distractor_id)
        ratio = (p_n + 1e-30) / (p_d + 1e-30)
        print(f"{name:6s} | p_needle={p_n:.3e} p_dist={p_d:.3e} ratio={ratio:.4g} | "
              f"rank_needle={r_n} rank_dist={r_d}")


=== banana vs mango ===
actual eviction distance: 536 | evicted: True
PLAIN  | p_needle=4.655e-10 p_dist=2.927e-08 ratio=0.01591 | rank_needle=142924 rank_dist=86620
J-LENS | p_needle=3.549e-11 p_dist=1.460e-09 ratio=0.02432 | rank_needle=148152 rank_dist=115025

=== Paris vs London ===
actual eviction distance: 536 | evicted: True
PLAIN  | p_needle=3.381e-07 p_dist=1.494e-08 ratio=22.63 | rank_needle=35079 rank_dist=99609
J-LENS | p_needle=8.744e-07 p_dist=4.050e-08 ratio=21.59 | rank_needle=7829 rank_dist=37478


#### C2 — Plain vs J-Lens diagnostic

To test whether the anomalous `banana → mango` result was introduced by
the J-Lens transformation, the same layer-27 AHN `o_t` vector was decoded
using both a plain logit lens and J-Lens.

At an actual eviction distance of 539 tokens:

| Pair | Plain ratio p(needle)/p(distractor) | J-Lens ratio |
|---|---:|---:|
| banana → mango | 0.015 | 0.024 |
| Paris → London | 23.33 | 21.55 |

For `banana → mango`, both readouts strongly favor the distractor.
For `Paris → London`, both strongly favor the true needle.

**Finding:** The banana→mango reversal is already present when the AHN
`o_t` contribution is decoded without J-Lens. J-Lens does not introduce
the direction of this anomaly.

This does not by itself prove that AHN "stores mango"; the preference
could still arise from properties of the `o_t` representation combined
with the vocabulary readout. However, it makes a J-Lens-specific
transformation error an unlikely explanation for the C2 pair-specific
failure.

In [22]:
# Diagnostic: is mango generally favored over banana by AHN o_t readout,
# even when the stored needle is NOT banana?
L, TEST_DISTANCE, TEST_FILLER = 27, 512, 0
banana_id, mango_id = encode1("banana"), encode1("mango")
test_needles = ["Paris", "Tokyo", "banana", "lantern"]

print("Stored needle | p(banana)/p(mango) | winner")
print("-" * 50)
for stored_needle in test_needles:
    scores, spec = score_word(stored_needle, {"banana": banana_id, "mango": mango_id},
                              TEST_DISTANCE, TEST_FILLER, layers=[L])
    assert spec["needle_is_evicted"], f"{stored_needle} was not actually evicted"
    p_banana = scores[L]["banana"][1]
    p_mango  = scores[L]["mango"][1]
    ratio = (p_banana + 1e-30) / (p_mango + 1e-30)
    winner = "banana" if ratio > 1 else "mango"
    print(f"{stored_needle:12s} | {ratio:17.6g} | {winner}")

Stored needle | p(banana)/p(mango) | winner
--------------------------------------------------
Paris        |         0.0203115 | mango
Tokyo        |         0.0205772 | mango
banana       |         0.0243151 | mango
lantern      |         0.0320833 | mango


#### C2 — Evidence of pair-specific baseline readout bias

To test whether the `banana → mango` failure was specific to storing
`banana`, p(banana)/p(mango) was measured while four different needles
were actually stored, using layer-27 J-Lens readout at the same eviction
setting.

| Stored needle | p(banana)/p(mango) |
|---|---:|
| Paris | 0.0207 |
| Tokyo | 0.0185 |
| banana | 0.0239 |
| lantern | 0.0323 |

`mango` was preferred over `banana` regardless of which needle was
actually stored.

**Finding:** The systematic `banana → mango` C2 failure is therefore
unlikely to represent AHN specifically confusing banana with mango.
Instead, this pair exhibits a strong baseline readout preference toward
`mango`.

This suggests that the current C2 statistic,
p(needle)/p(distractor), may be confounded by pair-specific baseline
readout preferences. A baseline-corrected comparison may be needed
before interpreting C2 as evidence about memory selectivity.

In [23]:
# Diagnostic: pair preference while varying the actually stored needle.
# Same layer/distance/filler/J-Lens for every comparison.
L, TEST_DISTANCE, TEST_FILLER = 27, 512, 0
pairs = {"Paris": "London", "Tokyo": "Osaka", "banana": "mango", "lantern": "torch"}
pair_ids = {n: (encode1(n), encode1(d)) for n, d in pairs.items()}
# flat target dict: every needle and distractor, read from each o_t
targets = {}
for n, (nid, did) in pair_ids.items():
    targets[f"n::{n}"] = nid
    targets[f"d::{n}"] = did

print("Stored      | Tested pair       | needle/dist ratio | winner")
print("-" * 68)
for stored in pairs:
    scores, spec = score_word(stored, targets, TEST_DISTANCE, TEST_FILLER, layers=[L])
    assert spec["needle_is_evicted"], f"{stored} was not actually evicted"
    s = scores[L]
    for needle, distractor in pairs.items():
        p_n = s[f"n::{needle}"][1]
        p_d = s[f"d::{needle}"][1]
        ratio = (p_n + 1e-30) / (p_d + 1e-30)
        winner = needle if ratio > 1 else distractor
        print(f"{stored:11s} | {needle:7s}/{distractor:7s} | {ratio:17.6g} | {winner}")
    print("-" * 68)

Stored      | Tested pair       | needle/dist ratio | winner
--------------------------------------------------------------------
Paris       | Paris  /London  |            21.587 | Paris
Paris       | Tokyo  /Osaka   |           6.35152 | Tokyo
Paris       | banana /mango   |         0.0203115 | mango
Paris       | lantern/torch   |            13.046 | lantern
--------------------------------------------------------------------
Tokyo       | Paris  /London  |           26.1988 | Paris
Tokyo       | Tokyo  /Osaka   |           6.87888 | Tokyo
Tokyo       | banana /mango   |         0.0205772 | mango
Tokyo       | lantern/torch   |           10.8047 | lantern
--------------------------------------------------------------------
banana      | Paris  /London  |           19.4836 | Paris
banana      | Tokyo  /Osaka   |           6.68386 | Tokyo
banana      | banana /mango   |         0.0243151 | mango
banana      | lantern/torch   |           9.68726 | lantern
------------------------------

#### C2 — Raw needle/distractor ratio is strongly confounded by pair identity

A cross-pair diagnostic was performed at layer 27. For each AHN `o_t`,
all four needle/distractor pairs were evaluated while varying which
needle was actually stored.

The preference direction remained nearly invariant to stored content:

- Paris > London: ~15–26×
- Tokyo > Osaka: ~6.5–7×
- mango > banana: ~31–54×
- lantern > torch: ~9–12.5×

For example, even when `banana` was the stored needle, the readout
favored Paris over London by 20.0×, Tokyo over Osaka by 6.63×,
mango over banana by ~41.8×, and lantern over torch by 9.40×.

**Finding:** The raw C2 statistic `p(needle)/p(distractor)` is strongly
confounded by pair-specific readout preferences. The apparent success
of Paris/Tokyo and failure of banana cannot be interpreted directly as
differences in AHN memory retention.

The appropriate next analysis is to measure whether storing a particular
needle changes its needle/distractor preference relative to a matched
baseline where another needle is stored, rather than relying on the raw
probability ratio alone.

In [24]:
 
# Rows = which needle was actually stored
# Columns = which pair is being tested
ratios = np.array([
    [21.5543, 6.49267, 0.0207231, 12.5071],  # stored Paris
    [26.0682, 7.01429, 0.0184562, 10.8719],  # stored Tokyo
    [20.0120, 6.62839, 0.0239433,  9.39896], # stored banana
    [15.4170, 6.53695, 0.0323120,  9.47369], # stored lantern
])

names = ["Paris", "Tokyo", "banana", "lantern"]

print("Needle   | when stored | baseline(other 3) | fold change")
print("-" * 64)

for i, name in enumerate(names):
    when_stored = ratios[i, i]

    # Baseline for this SAME pair when some other needle was stored.
    others = np.delete(ratios[:, i], i)

    # Geometric mean is appropriate because these are probability ratios.
    baseline = np.exp(np.mean(np.log(others)))

    fold_change = when_stored / baseline

    print(
        f"{name:8s} | "
        f"{when_stored:11.5g} | "
        f"{baseline:17.5g} | "
        f"{fold_change:11.4f}x"
    )

Needle   | when stored | baseline(other 3) | fold change
----------------------------------------------------------------
Paris    |      21.554 |            20.036 |      1.0758x
Tokyo    |      7.0143 |            6.5524 |      1.0705x
banana   |    0.023943 |           0.02312 |      1.0356x
lantern  |      9.4737 |            10.852 |      0.8730x


#### C2 — Baseline-corrected pair-baseline-corrected diagnostic effect

Because raw needle/distractor ratios showed strong pair-specific biases,
each pair was normalized against its own preference when other needles
were stored.

At layer 27 and the tested eviction setting:

| Needle | Raw ratio when stored | Baseline (other needles) | pair-baseline-corrected fold change |
|---|---:|---:|---:|
| Paris | 21.55 | 20.04 | 1.076× |
| Tokyo | 7.01 | 6.55 | 1.071× |
| banana | 0.0239 | 0.0231 | 1.036× |
| lantern | 9.47 | 10.85 | 0.873× |

Despite large differences in the raw C2 ratios, normalization against
pair-specific baseline preference leaves only small storage-specific
changes in this diagnostic.

**Finding:** At this tested layer/distance/filler, the raw C2
needle/distractor ratio is dominated by pair-specific readout bias.
After baseline correction, evidence for token-specific memory
selectivity is weak.

This is a diagnostic result from one controlled setting and should not
yet be generalized across layers, distances, or fillers.

In [25]:
# Inspection only — existing C2 rows.
# No inference and no modification of results.

c2_rows = [
    r for r in main
    if "p_distractor" in r
]

print("Total C2 rows:", len(c2_rows))
print("Needles:", sorted(set(r["needle"] for r in c2_rows)))
print("Layers:", sorted(set(r["layer"] for r in c2_rows)))
print(
    "Requested distances:",
    sorted(set(r["requested_distance"] for r in c2_rows))
)
print(
    "Filler indices:",
    sorted(set(r["filler_idx"] for r in c2_rows))
)

print("\nRows per layer / needle:")

counts = Counter(
    (r["layer"], r["needle"])
    for r in c2_rows
)

for key in sorted(counts):
    print(key, counts[key])

Total C2 rows: 252
Needles: ['Paris', 'Tokyo', 'banana', 'lantern']
Layers: [9, 18, 27]
Requested distances: [64, 256, 512, 1024, 2048, 4096, 8192]
Filler indices: [0, 1, 2]

Rows per layer / needle:
(9, 'Paris') 21
(9, 'Tokyo') 21
(9, 'banana') 21
(9, 'lantern') 21
(18, 'Paris') 21
(18, 'Tokyo') 21
(18, 'banana') 21
(18, 'lantern') 21
(27, 'Paris') 21
(27, 'Tokyo') 21
(27, 'banana') 21
(27, 'lantern') 21


In [26]:

# FULL C2 BASELINE-BIAS DIAGNOSTIC  (refactored: uses score_word; verified 1008 rows)
# 4 needles x 7 distances x 3 fillers = 84 forward passes; each reads ALL four pairs.
PAIRS = {"Paris": "London", "Tokyo": "Osaka", "banana": "mango", "lantern": "torch"}
pair_ids = {n: (encode1(n), encode1(d)) for n, d in PAIRS.items()}
targets = {}
for n, (nid, did) in pair_ids.items():
    targets[f"needle::{n}"] = nid
    targets[f"distr::{n}"]  = did

c2_bias_rows = []
total = len(PAIRS) * len(EXP["eviction_distances"]) * EXP["n_filler_variants"]
done = 0
for stored in PAIRS:
    for dist in EXP["eviction_distances"]:
        for f in range(EXP["n_filler_variants"]):
            scores, spec = score_word(stored, targets, dist, f)
            if scores is None:
                continue
            for L, s in scores.items():
                for tested, distractor in PAIRS.items():
                    rank_n, p_n = s[f"needle::{tested}"]
                    rank_d, p_d = s[f"distr::{tested}"]
                    c2_bias_rows.append({
                        "stored_needle": stored, "tested_needle": tested,
                        "distractor": distractor, "layer": L,
                        "requested_distance": dist,
                        "actual_eviction_distance": spec["actual_eviction_distance"],
                        "filler_idx": f,
                        "p_needle": p_n, "p_distractor": p_d,
                        "ratio": (p_n + 1e-30) / (p_d + 1e-30),
                        "rank_needle": rank_n, "rank_distractor": rank_d,
                    })
            done += 1
            if done % 10 == 0 or done == total:
                print(f"{done}/{total} forward passes completed")

df_bias = pd.DataFrame(c2_bias_rows)
print(f"\n=== SWEEP COMPLETE === rows: {len(df_bias)}")
print(df_bias.groupby(["layer", "stored_needle", "tested_needle"]).size().to_string())
ai.save_json({"rows": c2_bias_rows}, "04c1_rank_baseline.json")
print("saved -> 04c1_rank_baseline.json (%d rows)" % len(df_bias))


10/84 forward passes completed
20/84 forward passes completed
30/84 forward passes completed
40/84 forward passes completed
50/84 forward passes completed
60/84 forward passes completed
70/84 forward passes completed
80/84 forward passes completed
84/84 forward passes completed

=== SWEEP COMPLETE === rows: 1008
layer  stored_needle  tested_needle
9      Paris          Paris            21
                      Tokyo            21
                      banana           21
                      lantern          21
       Tokyo          Paris            21
                      Tokyo            21
                      banana           21
                      lantern          21
       banana         Paris            21
                      Tokyo            21
                      banana           21
                      lantern          21
       lantern        Paris            21
                      Tokyo            21
                      banana           21
                    

#### C1 — Rank-based baseline-corrected analysis

In [27]:
# C1 — rank-based baseline correction (Execution Tracker G2, 2026-09-08). NO GPU.
# Uses 04c1_rank_baseline.json. Compares rank when the tested needle WAS stored vs mean rank
# when a DIFFERENT needle was stored, matched on (layer, distance, filler).
# Stats helpers are GLOBAL. bootstrap_mean_ci is rank-space (additive). RNG stays local.
RNG = np.random.default_rng(42)

_c1_raw = json.load(open("results/run_3b_gdn/04c1_rank_baseline.json"))
d_c1 = pd.DataFrame(_c1_raw["rows"])
actual_c1  = d_c1[d_c1["stored_needle"] == d_c1["tested_needle"]].copy()
control_c1 = d_c1[d_c1["stored_needle"] != d_c1["tested_needle"]].copy()
control_mean_c1 = (control_c1.groupby(["layer","requested_distance","filler_idx","tested_needle"])
                   ["rank_needle"].mean().reset_index(name="control_mean_rank"))
merged_c1 = actual_c1.merge(control_mean_c1,
                            on=["layer","requested_distance","filler_idx","tested_needle"], how="inner")
merged_c1["delta_rank"] = merged_c1["rank_needle"] - merged_c1["control_mean_rank"]  # neg = better when stored
assert len(merged_c1) == 252
merged_c1["condition"] = list(zip(merged_c1["requested_distance"], merged_c1["filler_idx"]))
assert merged_c1["condition"].nunique() == 21
_cc = merged_c1.groupby(["layer","condition"]).size(); assert (_cc == 4).all(), _cc[_cc != 4]

# A. LAYER × NEEDLE
needle_results_c1 = []
for (layer, needle), g in merged_c1.groupby(["layer","tested_needle"]):
    x = g.sort_values(["requested_distance","filler_idx"])["delta_rank"].to_numpy()
    assert len(x) == 21
    mean_delta, lo, hi = bootstrap_mean_ci(x, RNG)
    needle_results_c1.append({"layer": layer, "needle": needle, "n_conditions": len(x),
                              "mean_delta_rank": mean_delta, "ci_low": lo, "ci_high": hi,
                              "p_raw": signflip_pvalue(x, RNG)})
needle_stats_c1 = pd.DataFrame(needle_results_c1)
needle_stats_c1["p_holm"] = holm_adjust(needle_stats_c1["p_raw"].to_numpy())
needle_stats_c1["significant_holm_005"] = needle_stats_c1["p_holm"] < 0.05
print("=== C1 MATCHED TEST -- LAYER x NEEDLE (Holm corrected across 12 tests) ===")
print(needle_stats_c1.sort_values(["layer","needle"]).to_string(index=False))

# B. POOLED BY LAYER — average the 4 needles within each layer×distance×filler first
clustered_c1 = (merged_c1.groupby(["layer","requested_distance","filler_idx"])["delta_rank"]
                .mean().reset_index(name="cluster_mean_delta"))
layer_results_c1 = []
for layer, g in clustered_c1.groupby("layer"):
    x = g.sort_values(["requested_distance","filler_idx"])["cluster_mean_delta"].to_numpy()
    assert len(x) == 21
    mean_delta, lo, hi = bootstrap_mean_ci(x, RNG)
    layer_results_c1.append({"layer": layer, "n_conditions": len(x),
                             "mean_delta_rank": mean_delta, "ci_low": lo, "ci_high": hi,
                             "p_raw": signflip_pvalue(x, RNG)})
layer_stats_c1 = pd.DataFrame(layer_results_c1)
layer_stats_c1["p_holm"] = holm_adjust(layer_stats_c1["p_raw"].to_numpy())
layer_stats_c1["significant_holm_005"] = layer_stats_c1["p_holm"] < 0.05
print("\n=== C1 MATCHED/CLUSTERED TEST -- POOLED BY LAYER (Holm corrected across 3 tests) ===")
print(layer_stats_c1.sort_values("layer").to_string(index=False))

ai.save_json({
    "by_needle": needle_stats_c1.to_dict(orient="records"),
    "by_layer": layer_stats_c1.to_dict(orient="records"),
    "readout": globals().get("READOUT", "unknown"),
    "lens_validated": globals().get("LENS_VALIDATED", None),
    "chance_rank": 151936 / 2,
}, "04d1_c1_rank_baseline_corrected.json")
print("\nsaved -> results/run_3b_gdn/04d1_c1_rank_baseline_corrected.json")

=== C1 MATCHED TEST -- LAYER x NEEDLE (Holm corrected across 12 tests) ===
 layer  needle  n_conditions  mean_delta_rank       ci_low     ci_high    p_raw   p_holm  significant_holm_005
     9   Paris            21      -585.444444 -1969.700794  510.620238 0.445376 1.000000                 False
     9   Tokyo            21      -678.047619 -1355.367460  -96.093254 0.048030 0.432266                 False
     9  banana            21      -291.507937  -724.448016  113.258333 0.205118 1.000000                 False
     9 lantern            21       482.746032  -323.704365 1485.628968 0.363086 1.000000                 False
    18   Paris            21     -1820.460317 -3139.752381 -490.120238 0.017830 0.178298                 False
    18   Tokyo            21      -537.904762 -1369.571825  303.671032 0.237738 1.000000                 False
    18  banana            21      -447.190476 -1889.944048  840.359524 0.549435 1.000000                 False
    18 lantern            21      -68

#### Needle-identity category test — place names vs common nouns (extends the C1 rank baseline)

In [28]:
# Needle-identity category test (place vs noun) -- refactored; verified 1512 rows
PLACE = ["Rome","Cairo","Moscow","Berlin","Madrid","Vienna","Athens","Dublin","Oslo","Lisbon","Warsaw","Prague"]
NOUN  = ["river","chair","window","garden","table","house","water","paper","stone","music","green","cloud"]
ALREADY_TESTED = {"Paris","Tokyo","banana","lantern"}

needles_by_category = []
for cat, words in [("place", PLACE), ("noun", NOUN)]:
    print(f"\n{cat}-name candidates:")
    for w in words:
        if w in ALREADY_TESTED:
            continue
        ids = tok.encode(f" {w}", add_special_tokens=False)
        if len(ids) != 1:
            print(f"  skip {w!r}: {len(ids)} tokens"); continue
        print(f"  {w:10s} -> {ids[0]}")
        needles_by_category.append((w, ids[0], cat))
n_place = sum(c == "place" for _, _, c in needles_by_category)
print(f"\n{len(needles_by_category)} needles total ({n_place} place, {len(needles_by_category)-n_place} noun)")

category_rows = []
total = len(needles_by_category) * len(EXP["eviction_distances"]) * EXP["n_filler_variants"]
done = 0
for word, word_id, cat in needles_by_category:
    for dist in EXP["eviction_distances"]:
        for f in range(EXP["n_filler_variants"]):
            scores, spec = score_word(word, {"n": word_id}, dist, f)
            if scores is None:
                continue
            for L, s in scores.items():
                category_rows.append({
                    "needle": word, "category": cat, "layer": L,
                    "requested_distance": dist,
                    "actual_eviction_distance": spec["actual_eviction_distance"],
                    "filler_idx": f, "rank_needle": s["n"][0],
                })
            done += 1
            if done % 20 == 0 or done == total:
                print(f"{done}/{total} forward passes completed")
    ai.free_cuda()

df_category = pd.DataFrame(category_rows)
print("rows collected:", len(df_category))
ai.save_json({"rows": category_rows}, "04e_needle_category_extended.json")
print("saved -> 04e_needle_category_extended.json")



place-name candidates:
  Rome       -> 21718
  Cairo      -> 52550
  Moscow     -> 22415
  Berlin     -> 19846
  Madrid     -> 24081
  Vienna     -> 46287
  Athens     -> 45826
  Dublin     -> 32877
  Oslo       -> 57858
  Lisbon     -> 80701
  Warsaw     -> 72176
  Prague     -> 67289

noun-name candidates:
  river      -> 14796
  chair      -> 10496
  window     -> 3241
  garden     -> 13551
  table      -> 1965
  house      -> 3753
  water      -> 3015
  paper      -> 5567
  stone      -> 9798
  music      -> 4627
  green      -> 6176
  cloud      -> 9437

24 needles total (12 place, 12 noun)
20/504 forward passes completed
40/504 forward passes completed
60/504 forward passes completed
80/504 forward passes completed
100/504 forward passes completed
120/504 forward passes completed
140/504 forward passes completed
160/504 forward passes completed
180/504 forward passes completed
200/504 forward passes completed
220/504 forward passes completed
240/504 forward passes completed
260/

#### Needle-identity category test — statistical comparison

In [29]:
# Tests whether the place-name/common-noun split (Paris/Tokyo vs banana/lantern at layer 27)
# generalizes to a larger needle set, or was specific to those 4 words. NO GPU.
# Between-needle comparison (each needle in one category) -> two-sample permutation test on
# needle-level mean rank. perm_test_two_sample / holm_adjust are GLOBAL; RNG stays local.
RNG = np.random.default_rng(42)

_cat_raw = json.load(open("results/run_3b_gdn/04e_needle_category_extended.json"))
d_cat = pd.DataFrame(_cat_raw["rows"])
needle_means = (d_cat.groupby(["layer", "needle", "category"])["rank_needle"]
                .mean().reset_index(name="mean_rank"))

results = []
for L, g in needle_means.groupby("layer"):
    place = g[g["category"] == "place"]["mean_rank"].to_numpy()
    noun  = g[g["category"] == "noun"]["mean_rank"].to_numpy()
    diff, p = perm_test_two_sample(place, noun, RNG)
    results.append({"layer": L, "n_place": len(place), "n_noun": len(noun),
                    "mean_rank_place": float(place.mean()), "mean_rank_noun": float(noun.mean()),
                    "diff_place_minus_noun": diff, "p_raw": p})
stats = pd.DataFrame(results)
stats["p_holm"] = holm_adjust(stats["p_raw"].to_numpy())
stats["significant_holm_005"] = stats["p_holm"] < 0.05
print("=== PLACE vs NOUN mean rank, by layer (Holm corrected across 3 tests) ===")
print(stats.to_string(index=False))
print("\nPer-needle means (for the write-up / sanity check):")
print(needle_means.sort_values(["layer", "category", "mean_rank"]).to_string(index=False))

ai.save_json({"by_layer": stats.to_dict(orient="records"),
              "by_needle": needle_means.to_dict(orient="records")},
             "04f_needle_category_stats.json")
print("\nsaved -> results/run_3b_gdn/04f_needle_category_stats.json")

=== PLACE vs NOUN mean rank, by layer (Holm corrected across 3 tests) ===
 layer  n_place  n_noun  mean_rank_place  mean_rank_noun  diff_place_minus_noun    p_raw   p_holm  significant_holm_005
     9       12      12     82500.876984    92403.432540           -9902.555556 0.481925 0.481925                 False
    18       12      12     72021.912698    89368.460317          -17346.547619 0.008500 0.017000                  True
    27       12      12     24665.706349    92829.984127          -68164.277778 0.000170 0.000510                  True

Per-needle means (for the write-up / sanity check):
 layer needle category     mean_rank
     9 garden     noun  32553.761905
     9  paper     noun  50582.809524
     9  chair     noun  58085.142857
     9  house     noun  82431.000000
     9  green     noun  86330.666667
     9  table     noun  88507.476190
     9 window     noun  99889.809524
     9  stone     noun 103794.190476
     9  cloud     noun 122882.142857
     9  water     noun 

#### Needle-identity category test — is it "place" or "proper noun"? (adds person names)

In [30]:
# Person-name category sweep -- refactored; proper nouns that aren't places
PERSON = ["John","Mary","James","Sarah","Michael","Emma","David","Laura","Robert","Anna","Peter","Linda"]
ALREADY_TESTED = {"Paris","Tokyo","banana","lantern",
                  "Rome","Cairo","Moscow","Berlin","Madrid","Vienna","Athens","Dublin","Oslo","Lisbon","Warsaw","Prague",
                  "river","chair","window","garden","table","house","water","paper","stone","music","green","cloud"}

print("person-name candidates:")
person_needles = []
for w in PERSON:
    if w in ALREADY_TESTED:
        continue
    ids = tok.encode(f" {w}", add_special_tokens=False)
    if len(ids) != 1:
        print(f"  skip {w!r}: {len(ids)} tokens"); continue
    print(f"  {w:10s} -> {ids[0]}")
    person_needles.append((w, ids[0]))
print("person needles:", len(person_needles))

person_rows = []
total = len(person_needles) * len(EXP["eviction_distances"]) * EXP["n_filler_variants"]
done = 0
for word, word_id in person_needles:
    for dist in EXP["eviction_distances"]:
        for f in range(EXP["n_filler_variants"]):
            scores, spec = score_word(word, {"n": word_id}, dist, f)
            if scores is None:
                continue
            for L, s in scores.items():
                person_rows.append({
                    "needle": word, "category": "person", "layer": L,
                    "requested_distance": dist,
                    "actual_eviction_distance": spec["actual_eviction_distance"],
                    "filler_idx": f, "rank_needle": s["n"][0],
                })
            done += 1
            if done % 20 == 0 or done == total:
                print(f"{done}/{total} forward passes completed")
    ai.free_cuda()

df_person = pd.DataFrame(person_rows)
print("rows collected:", len(df_person))
ai.save_json({"rows": person_rows}, "04g_needle_category_person_extended.json")
print("saved -> 04g_needle_category_person_extended.json")


person-name candidates:
  John       -> 3757
  Mary       -> 10244
  James      -> 7801
  Sarah      -> 20445
  Michael    -> 7937
  Emma       -> 34935
  David      -> 6798
  Laura      -> 29828
  Robert     -> 8397
  Anna       -> 23223
  Peter      -> 11044
  Linda      -> 38062
person needles: 12
20/252 forward passes completed
40/252 forward passes completed
60/252 forward passes completed
80/252 forward passes completed
100/252 forward passes completed
120/252 forward passes completed
140/252 forward passes completed
160/252 forward passes completed
180/252 forward passes completed
200/252 forward passes completed
220/252 forward passes completed
240/252 forward passes completed
252/252 forward passes completed
rows collected: 756
saved -> 04g_needle_category_person_extended.json


#### Three-way category comparison — place vs. person vs. common noun

In [31]:
# Combines all three needle sources into one three-way test. NO GPU -- reads
# 04c1_rank_baseline.json (original 4), 04e (12 place + 12 noun), 04g (person names).
# Pairwise permutation tests (place/person/noun), Holm-corrected across 3 comparisons x 3 layers.
# perm_test_two_sample / holm_adjust are GLOBAL; RNG stays local.
RNG = np.random.default_rng(42)

# original 4 needles: stored==tested rows from the C1 rank sweep
_orig_raw = json.load(open("results/run_3b_gdn/04c1_rank_baseline.json"))
d_orig = pd.DataFrame(_orig_raw["rows"])
d_orig = d_orig[d_orig["stored_needle"] == d_orig["tested_needle"]].copy().rename(columns={"stored_needle": "needle"})
ORIG_CATEGORY = {"Paris": "place", "Tokyo": "place", "banana": "noun", "lantern": "noun"}
d_orig["category"] = d_orig["needle"].map(ORIG_CATEGORY)
d_orig = d_orig[["needle", "category", "layer", "rank_needle"]]

# 24 new place/noun needles
d_cat = pd.DataFrame(json.load(open("results/run_3b_gdn/04e_needle_category_extended.json"))["rows"]
                     )[["needle", "category", "layer", "rank_needle"]]
# person needles
d_person = pd.DataFrame(json.load(open("results/run_3b_gdn/04g_needle_category_person_extended.json"))["rows"]
                        )[["needle", "category", "layer", "rank_needle"]]

d_all = pd.concat([d_orig, d_cat, d_person], ignore_index=True)
needle_means = (d_all.groupby(["layer", "needle", "category"])["rank_needle"]
                .mean().reset_index(name="mean_rank"))
print("n needles per category:")
print(needle_means[needle_means["layer"] == 9].groupby("category").size().to_string())

PAIRS = [("place", "person"), ("place", "noun"), ("person", "noun")]
results = []
for L, g in needle_means.groupby("layer"):
    by_cat = {c: g[g["category"] == c]["mean_rank"].to_numpy() for c in ("place", "person", "noun")}
    for cat_a, cat_b in PAIRS:
        diff, p = perm_test_two_sample(by_cat[cat_a], by_cat[cat_b], RNG)
        results.append({"layer": L, "comparison": f"{cat_a}_vs_{cat_b}",
                        "n_a": len(by_cat[cat_a]), "n_b": len(by_cat[cat_b]),
                        "mean_a": float(by_cat[cat_a].mean()), "mean_b": float(by_cat[cat_b].mean()),
                        "diff": diff, "p_raw": p})
cat_stats = pd.DataFrame(results)
cat_stats["p_holm"] = holm_adjust(cat_stats["p_raw"].to_numpy())
cat_stats["significant_holm_005"] = cat_stats["p_holm"] < 0.05
print("\n=== THREE-WAY CATEGORY COMPARISON (Holm corrected across 9 tests) ===")
print(cat_stats.to_string(index=False))
print("\nCategory means by layer:")
print(needle_means.groupby(["layer", "category"])["mean_rank"].mean().reset_index().to_string(index=False))

ai.save_json({"pairwise": cat_stats.to_dict(orient="records"),
              "by_needle": needle_means.to_dict(orient="records")},
             "04h_needle_category_person_stats.json")
print("\nsaved -> results/run_3b_gdn/04h_needle_category_person_stats.json")

n needles per category:
category
noun      14
person    12
place     14

=== THREE-WAY CATEGORY COMPARISON (Holm corrected across 9 tests) ===
 layer      comparison  n_a  n_b       mean_a       mean_b          diff    p_raw   p_holm  significant_holm_005
     9 place_vs_person   14   12 78069.448980 82784.551587  -4715.102608 0.709773 1.000000                 False
     9   place_vs_noun   14   14 78069.448980 93877.299320 -15807.850340 0.214998 0.870711                 False
     9  person_vs_noun   12   14 82784.551587 93877.299320 -11092.747732 0.345147 1.000000                 False
    18 place_vs_person   14   12 76340.139456 85028.392857  -8688.253401 0.162238 0.870711                 False
    18   place_vs_noun   14   14 76340.139456 87999.401361 -11659.261905 0.093569 0.654983                 False
    18  person_vs_noun   12   14 85028.392857 87999.401361  -2971.008503 0.674843 1.000000                 False
    27 place_vs_person   14   12 23115.653061 12303.075397  10812.

In [32]:
# FULL C2 baseline-corrected analysis. Uses df_bias. No inference, no official-result writes.
# Log-ratio space: log[p(needle)/p(distractor)]. For each layer×distance×filler×pair, compare
# the ratio when THAT needle was stored vs the mean when the other 3 were stored.
work = df_bias.copy()
EPS = 1e-30
work["log_ratio"] = np.log((work["p_needle"] + EPS) / (work["p_distractor"] + EPS))

corrected = []
for (layer, distance, filler, tested), g in work.groupby(
        ["layer", "requested_distance", "filler_idx", "tested_needle"]):
    target   = g[g["stored_needle"] == tested]
    baseline = g[g["stored_needle"] != tested]
    assert len(target) == 1, (layer, distance, filler, tested, len(target))
    assert len(baseline) == 3, (layer, distance, filler, tested, len(baseline))
    target_log   = float(target["log_ratio"].iloc[0])
    baseline_log = float(baseline["log_ratio"].mean())
    delta_log    = target_log - baseline_log
    corrected.append({
        "layer": layer, "requested_distance": distance, "filler_idx": filler, "needle": tested,
        "target_ratio": float(np.exp(target_log)),

Corrected observations: 252

=== FULL BASELINE-CORRECTED C2 ===
                n  median_corrected_fold  geometric_mean_fold  frac_above_1
layer needle                                                               
9     Paris    21               1.005833             0.994966      0.523810
      Tokyo    21               1.014250             1.013177      0.571429
      banana   21               1.008566             1.008744      0.714286
      lantern  21               0.990841             0.974039      0.476190
18    Paris    21               0.996093             0.996470      0.476190
      Tokyo    21               0.999445             1.020077      0.476190
      banana   21               1.018265             1.021613      0.619048
      lantern  21               1.191236             1.128823      0.666667
27    Paris    21               1.088454             1.088755      0.809524
      Tokyo    21               1.052358             1.025084      0.666667
      banana   21       

#### C2 — Full matched baseline-corrected analysis

The pair-specific readout-bias diagnostic was extended across the full
C2 design: 3 layers × 7 eviction distances × 3 filler variants ×
4 needle/distractor pairs (252 matched corrected observations).

For every layer × distance × filler × tested-pair condition, the
needle/distractor log-probability ratio when the tested needle was
actually stored was compared with the same pair's mean log-ratio when
one of the other three needles was stored.

Pooled descriptive results:

| Layer | Median corrected fold | Geometric mean fold | Fraction > 1 |
|---|---:|---:|---:|
| 9  | 1.004 | 1.000 | 52.4% |
| 18 | 1.034 | 1.045 | 61.9% |
| 27 | 1.014 | 0.990 | 54.8% |

These pooled values are descriptive only because the 84 observations
within each layer are not independent.

**Finding:** The large raw differences observed in C2 are strongly
affected by pair-specific readout preferences. After matching each pair
against its own baseline, Layers 9 and 27 remain close to 1×, while
Layer 18 shows a small positive corrected association.

Statistical interpretation is deferred to the condition-level analysis
below, which aggregates the four tested pairs into 21 condition-level
observations per layer.

This result concerns the validity and interpretability of the current
C2 readout statistic. It does not establish that AHN contains no
token-specific information, because such information may not be
recoverable by the current J-Lens/vocabulary readout.

In [33]:
# Leakage check: inspect whether C2 target/distractor words
# accidentally appear in prompts where they should not.

WORDS = [
    "Paris", "London",
    "Tokyo", "Osaka",
    "banana", "mango",
    "lantern", "torch",
]

for stored in ["Paris", "Tokyo", "banana", "lantern"]:

    spec = ai.build_niah_prompt(
        tok,
        stored,
        bundle,
        eviction_distance=512,
        in_window=False,
        filler_idx=0,
    )

    prompt = spec["prompt"]

    print(f"\n=== STORED: {stored} ===")

    for word in WORDS:
        count = prompt.lower().count(word.lower())

        if count:
            print(f"{word:8s}: {count}")


=== STORED: Paris ===
Paris   : 1

=== STORED: Tokyo ===
Tokyo   : 1

=== STORED: banana ===
banana  : 1

=== STORED: lantern ===
lantern : 1


In [34]:
# FULL C2 prompt-leakage check
# 4 needles × 7 distances × 3 fillers = 84 prompts
# NO model inference / NO GPU / modifies nothing.

WORDS = [
    "Paris", "London",
    "Tokyo", "Osaka",
    "banana", "mango",
    "lantern", "torch",
]

STORED = ["Paris", "Tokyo", "banana", "lantern"]

leaks = []
checked = 0

for stored in STORED:
    for distance in EXP["eviction_distances"]:
        for filler_idx in range(EXP["n_filler_variants"]):

            spec = ai.build_niah_prompt(
                tok,
                stored,
                bundle,
                eviction_distance=distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            prompt_lower = spec["prompt"].lower()
            checked += 1

            for word in WORDS:
                count = prompt_lower.count(word.lower())

                # Intended stored needle should appear exactly once.
                if word == stored:
                    if count != 1:
                        leaks.append({
                            "stored": stored,
                            "distance": distance,
                            "filler": filler_idx,
                            "word": word,
                            "count": count,
                            "problem": "stored needle count != 1",
                        })

                # Every other C2 word should be absent.
                elif count != 0:
                    leaks.append({
                        "stored": stored,
                        "distance": distance,
                        "filler": filler_idx,
                        "word": word,
                        "count": count,
                        "problem": "unexpected word in prompt",
                    })

print("Prompts checked:", checked)
print("Problems found:", len(leaks))

if leaks:
    for x in leaks:
        print(x)
else:
    print("PASS: no C2 target/distractor contamination detected.")

Prompts checked: 84
Problems found: 0
PASS: no C2 target/distractor contamination detected.


#### C2 — Prompt contamination check

All 84 prompts used in the full C2 diagnostic
(4 needles × 7 distances × 3 filler variants) were checked for
accidental occurrences of all C2 needle and distractor words.

Each prompt contained its intended stored needle exactly once and
contained none of the other tested needles or distractors.

- Prompts checked: 84
- Contamination cases: 0

**Finding:** The observed pair-specific C2 readout preferences cannot
be explained by accidental target/distractor word contamination in the
generated prompts.

This rules out this specific form of prompt-level leakage, but does not
rule out every possible source of experimental bias or leakage.

In [35]:
# C2 — matched/clustered statistical validation (NO GPU; uses df_corrected). Log-space.
# Helpers are GLOBAL (defined in setup). RNG stays local so numbers match the original.
RNG = np.random.default_rng(42)
d = df_corrected.copy()
d["condition"] = list(zip(d["requested_distance"], d["filler_idx"]))
assert len(d) == 252
assert d["condition"].nunique() == 21
_c = d.groupby(["layer", "condition"]).size(); assert (_c == 4).all(), _c[_c != 4]

# A. LAYER × NEEDLE
needle_results = []
for (layer, needle), g in d.groupby(["layer", "needle"]):
    x = g.sort_values(["requested_distance", "filler_idx"])["delta_log_ratio"].to_numpy()
    assert len(x) == 21
    fold, lo, hi = bootstrap_mean_log_ci(x, RNG)
    needle_results.append({"layer": layer, "needle": needle, "n_conditions": len(x),
                           "geometric_mean_fold": fold, "ci_low": lo, "ci_high": hi,
                           "p_raw": signflip_pvalue(x, RNG)})
needle_stats = pd.DataFrame(needle_results)
needle_stats["p_holm"] = holm_adjust(needle_stats["p_raw"].to_numpy())
needle_stats["significant_holm_005"] = needle_stats["p_holm"] < 0.05
print("=== MATCHED TEST — LAYER × NEEDLE (Holm corrected across 12 tests) ===")
print(needle_stats.sort_values(["layer", "needle"]).to_string(index=False))

# B. POOLED BY LAYER — average the 4 needles within each layer×distance×filler first
clustered = (d.groupby(["layer", "requested_distance", "filler_idx"])["delta_log_ratio"]
             .mean().reset_index(name="cluster_mean_log"))
layer_results = []
for layer, g in clustered.groupby("layer"):
    x = g.sort_values(["requested_distance", "filler_idx"])["cluster_mean_log"].to_numpy()
    assert len(x) == 21
    fold, lo, hi = bootstrap_mean_log_ci(x, RNG)
    layer_results.append({"layer": layer, "n_conditions": len(x),
                          "geometric_mean_fold": fold, "ci_low": lo, "ci_high": hi,
                          "p_raw": signflip_pvalue(x, RNG)})
layer_stats = pd.DataFrame(layer_results)
layer_stats["p_holm"] = holm_adjust(layer_stats["p_raw"].to_numpy())
layer_stats["significant_holm_005"] = layer_stats["p_holm"] < 0.05
print("\n=== MATCHED/CLUSTERED TEST — POOLED BY LAYER (Holm corrected across 3 tests) ===")
print(layer_stats.sort_values("layer").to_string(index=False))

=== MATCHED TEST — LAYER × NEEDLE (Holm corrected across 12 tests) ===
 layer  needle  n_conditions  geometric_mean_fold   ci_low  ci_high    p_raw   p_holm  significant_holm_005
     9   Paris            21             0.994966 0.975568 1.014507 0.626234 1.000000                 False
     9   Tokyo            21             1.013177 0.988070 1.036058 0.298667 1.000000                 False
     9  banana            21             1.008744 0.997105 1.020372 0.165848 1.000000                 False
     9 lantern            21             0.974039 0.937749 1.005587 0.177888 1.000000                 False
    18   Paris            21             0.996470 0.925459 1.066372 0.924331 1.000000                 False
    18   Tokyo            21             1.020077 0.972504 1.077232 0.475365 1.000000                 False
    18  banana            21             1.021613 0.978052 1.068827 0.369776 1.000000                 False
    18 lantern            21             1.128823 1.042792 1.2248

In [36]:
# CORRECTED CONDITION-LEVEL ANALYSIS. Unit = (distance × filler); 4 needles averaged per condition.
# Expected: 7 distances × 3 fillers = 21 observations/layer. Uses global summarize_effect.
required = {"layer", "requested_distance", "filler_idx", "delta_log_ratio"}
missing = required - set(df_corrected.columns)
assert not missing, f"Missing columns: {missing}"

cond = (df_corrected.groupby(["layer", "requested_distance", "filler_idx"], as_index=False)
        .agg(delta_log_ratio=("delta_log_ratio", "mean"), n_needles=("delta_log_ratio", "size")))
print("=== SANITY CHECK ===")
print("Original rows:", len(df_corrected), "| Condition rows:", len(cond))
print("\nConditions per layer:"); print(cond.groupby("layer").size())
print("\nNeedles per condition:"); print(cond["n_needles"].value_counts().sort_index())
assert (cond["n_needles"] == 4).all(), "ERROR: Some conditions do not contain exactly 4 needles."

results = pd.DataFrame([{**summarize_effect(g["delta_log_ratio"]), "layer": layer}
                        for layer, g in cond.groupby("layer")])
# rename to this cell's column names, reorder
results = results.rename(columns={"mean_log": "mean_log_effect", "fold": "geom_fold", "n": "n_conditions"})[
    ["layer", "n_conditions", "mean_log_effect", "geom_fold", "CI_low", "CI_high", "t", "p"]]
print("\n=== CORRECTED n=21 ANALYSIS ===")
print(results.to_string(index=False, float_format=lambda x: f"{x:.6g}"))

ModuleNotFoundError: No module named 'scipy'

### ⚠️ Deprecated pooled analysis — retained for audit trail

The following cell is retained only to document the original analysis and **should not be used for statistical inference or final conclusions**.

It treats all 84 needle-level observations per layer as independent (`n = 84`). However, the four needle measurements within each `(layer × requested_distance × filler_idx)` condition are not independent replicates. This results in pseudoreplication.

The corrected analysis averages across the four needles within each `(layer × requested_distance × filler_idx)` condition, producing:

- **21 condition-level observations per layer**
- 7 requested distances × 3 filler variants = 21 conditions
- 4 needles averaged within each condition

The corrected condition-level analysis supersedes the pooled results below.

#### Corrected results

| Layer | n | Geometric-mean fold | 95% CI | p-value |
|------:|---:|--------------------:|:------:|--------:|
| 9  | 21 | 1.000× | [0.989, 1.012] | 0.948 |
| 18 | 21 | 1.045× | [1.019, 1.072] | 0.00183 |
| 27 | 21 | 0.990× | [0.965, 1.017] | 0.455 |

Layer 9 and Layer 27 are consistent with no aggregate corrected effect.

Layer 18 shows a small positive corrected association in the condition-level analysis. This association is examined further in the robustness and scrambled-content controls below before any memory-specific interpretation is made.

All results remain conditional on the J-Lens not having passed the full Table 3 validation battery.

In [37]:
# ROBUSTNESS BATTERY FOR CORRECTED C2 (NO GPU). (1) n=21 condition-level; (2) leave-one-distance-out;
# (3) leave-one-filler-out; (4) per-needle; (5) fixed blocked permutation test.
# summarize_effect is GLOBAL. RNG stays local (used by the blocked permutation below).
RNG = np.random.default_rng(42)
N_PERM = 20_000
_ff = lambda x: f"{x:.6g}"

needle_col = next((c for c in ["needle", "stored_needle", "target_needle", "needle_word"]
                   if c in df_corrected.columns), None)
assert needle_col, f"No needle column in {list(df_corrected.columns)}"
print("Using needle column:", needle_col)

# 1. CONDITION-LEVEL
cond = (df_corrected.groupby(["layer", "requested_distance", "filler_idx"], as_index=False)
        .agg(delta_log_ratio=("delta_log_ratio", "mean"), n_needles=("delta_log_ratio", "size")))
assert (cond["n_needles"] == 4).all(), "Some conditions do not contain exactly 4 needles."
print("\n" + "=" * 70 + "\n1. CORRECTED CONDITION-LEVEL ANALYSIS\n" + "=" * 70)
print("Original rows:", len(df_corrected), "| Condition rows:", len(cond))
print("Conditions per layer:"); print(cond.groupby("layer").size())
baseline = pd.DataFrame([{**summarize_effect(g["delta_log_ratio"]), "layer": layer}
                         for layer, g in cond.groupby("layer")])[
    ["layer", "n", "mean_log", "fold", "CI_low", "CI_high", "t", "p"]]
print("\n" + baseline.to_string(index=False, float_format=_ff))

# 2. LEAVE-ONE-DISTANCE-OUT
print("\n" + "=" * 70 + "\n2. LEAVE-ONE-DISTANCE-OUT\n" + "=" * 70)
lodo_rows = []
for layer in sorted(cond["layer"].unique()):
    ldf = cond[cond["layer"] == layer]
    for dropped in sorted(ldf["requested_distance"].unique()):
        g = ldf[ldf["requested_distance"] != dropped]
        lodo_rows.append({"layer": layer, "dropped_distance": dropped, **summarize_effect(g["delta_log_ratio"])})
lodo = pd.DataFrame(lodo_rows)
print(lodo[["layer", "dropped_distance", "n", "fold", "CI_low", "CI_high", "p"]].to_string(index=False, float_format=_ff))

# 3. LEAVE-ONE-FILLER-OUT
print("\n" + "=" * 70 + "\n3. LEAVE-ONE-FILLER-OUT\n" + "=" * 70)
lofo_rows = []
for layer in sorted(cond["layer"].unique()):
    ldf = cond[cond["layer"] == layer]
    for dropped in sorted(ldf["filler_idx"].unique()):
        g = ldf[ldf["filler_idx"] != dropped]
        lofo_rows.append({"layer": layer, "dropped_filler": dropped, **summarize_effect(g["delta_log_ratio"])})
lofo = pd.DataFrame(lofo_rows)
print(lofo[["layer", "dropped_filler", "n", "fold", "CI_low", "CI_high", "p"]].to_string(index=False, float_format=_ff))

# 4. PER-NEEDLE
print("\n" + "=" * 70 + "\n4. PER-NEEDLE ANALYSIS\n" + "=" * 70)
needle_rows = [{"layer": layer, "needle": needle, **summarize_effect(g["delta_log_ratio"])}
               for (layer, needle), g in df_corrected.groupby(["layer", needle_col])]
needle_results = pd.DataFrame(needle_rows)
print(needle_results[["layer", "needle", "n", "fold", "CI_low", "CI_high", "p"]]
      .sort_values(["layer", "needle"]).to_string(index=False, float_format=_ff))

# 5. BLOCKED PERMUTATION TEST
print("\n" + "=" * 70 + "\n5. BLOCKED PERMUTATION TEST\n" + "=" * 70)
required_bias = {"stored_needle", "tested_needle", "distractor", "layer",
                 "requested_distance", "filler_idx", "p_needle", "p_distractor"}
missing = required_bias - set(df_bias.columns)
assert not missing, f"Missing df_bias columns: {missing}"
bias = df_bias.copy()
min_prob = min(bias["p_needle"].min(), bias["p_distractor"].min())
print("Minimum probability:", min_prob)
if min_prob <= 0:
    raise ValueError("Zero/negative probability found. Cannot compute safe log-ratios.")
bias["raw_log_ratio"] = np.log(bias["p_needle"]) - np.log(bias["p_distractor"])

block_cols = ["layer", "requested_distance", "filler_idx", "tested_needle", "distractor"]
block_sizes = bias.groupby(block_cols).size()
print("\nBlock-size counts:"); print(block_sizes.value_counts().sort_index())
bad_blocks = block_sizes[block_sizes != 4]
if len(bad_blocks) > 0:
    print("\nBad blocks:"); print(bad_blocks.head(20))
    raise ValueError(f"{len(bad_blocks)} blocks do not contain exactly 4 stored prompts.")
print("All matched blocks contain exactly 4 stored prompts: PASS")

blocks = []
for key, g in bias.groupby(block_cols):
    g = g.sort_values("stored_needle")
    blocks.append({"layer": key[0], "distance": key[1], "filler": key[2],
                   "tested_needle": key[3], "distractor": key[4],
                   "vals": g["raw_log_ratio"].to_numpy(dtype=float),
                   "stored_order": g["stored_needle"].tolist()})

all_orders = {tuple(b["stored_order"]) for b in blocks}
print("\nUnique stored-needle orders:", all_orders)
if len(all_orders) != 1:
    raise ValueError("Stored-needle ordering is not consistent across blocks.")
print("Stored-needle ordering consistent: PASS")

observed = cond.groupby("layer")["delta_log_ratio"].mean().to_dict()
print("\nObserved corrected effects:")
for layer, obs in observed.items():
    print(f"Layer {layer}: mean_log={obs:.6f}, fold={np.exp(obs):.6f}x")

perm_results = []
for layer in sorted(cond["layer"].unique()):
    layer_blocks = [b for b in blocks if b["layer"] == layer]
    condition_keys = sorted({(b["distance"], b["filler"]) for b in layer_blocks})
    print(f"\nLayer {layer}: {len(layer_blocks)} matched blocks, {len(condition_keys)} conditions")
    assert len(condition_keys) == 21, f"Expected 21 conditions at layer {layer}; found {len(condition_keys)}"
    null_stats = np.empty(N_PERM, dtype=float)
    for perm_i in range(N_PERM):
        condition_effects = []
        for distance, filler in condition_keys:
            these = sorted([b for b in layer_blocks if b["distance"] == distance and b["filler"] == filler],
                           key=lambda b: (b["tested_needle"], b["distractor"]))
            if len(these) != 4:
                raise ValueError(f"Expected 4 tested-pair blocks for layer={layer}, "
                                 f"distance={distance}, filler={filler}; found {len(these)}")
            assignment = RNG.permutation(4)
            pair_effects = [b["vals"][assignment[j]] - np.delete(b["vals"], assignment[j]).mean()
                            for j, b in enumerate(these)]
            condition_effects.append(np.mean(pair_effects))
        null_stats[perm_i] = np.mean(condition_effects)
    obs = observed[layer]
    p_perm = (np.sum(np.abs(null_stats) >= abs(obs)) + 1) / (N_PERM + 1)
    perm_results.append({"layer": layer, "observed_mean_log": obs, "observed_fold": np.exp(obs),
                         "null_mean": np.mean(null_stats), "null_sd": np.std(null_stats, ddof=1),
                         "permutation_p": p_perm})
perm_results = pd.DataFrame(perm_results)
print("\n" + "=" * 70 + "\nBLOCKED PERMUTATION RESULTS\n" + "=" * 70)
print(perm_results.to_string(index=False, float_format=_ff))

# FINAL ROBUSTNESS SUMMARY
print("\n" + "=" * 70 + "\nFINAL ROBUSTNESS SUMMARY\n" + "=" * 70)
for layer in sorted(cond["layer"].unique()):
    base = baseline[baseline["layer"] == layer].iloc[0]
    ld = lodo[lodo["layer"] == layer]; lf = lofo[lofo["layer"] == layer]
    nd = needle_results[needle_results["layer"] == layer]
    pp = perm_results[perm_results["layer"] == layer].iloc[0]
    print(f"\nLAYER {layer}")
    print(f"  Main corrected fold:    {base['fold']:.4f}x")
    print(f"  Main 95% CI:            [{base['CI_low']:.4f}, {base['CI_high']:.4f}]")
    print(f"  Main t-test p:          {base['p']:.6g}")
    print(f"  Leave-distance folds:   {ld['fold'].min():.4f}x to {ld['fold'].max():.4f}x")
    print(f"  Leave-filler folds:     {lf['fold'].min():.4f}x to {lf['fold'].max():.4f}x")
    print(f"  Per-needle folds:       {nd['fold'].min():.4f}x to {nd['fold'].max():.4f}x")
    print(f"  Block permutation p:    {pp['permutation_p']:.6g}")
print("\nDONE.")

Using needle column: needle

1. CORRECTED CONDITION-LEVEL ANALYSIS
Original rows: 252 | Condition rows: 63
Conditions per layer:
layer
9     21
18    21
27    21
dtype: int64


NameError: name 'summarize_effect' is not defined

In [ ]:
# C2 EXPANDED LEAKAGE / PROMPT-CONFOUND CHECKS (CPU only). Checks A-F: distance/filler balance,
# eviction distance, needle position, token length, local context. Ruling out prompt confounds.
print("=" * 72 + "\nC2 EXPANDED LEAKAGE / PROMPT-CONFOUND CHECKS\n" + "=" * 72)
print("\ndf_bias shape:", df_bias.shape, "\ncolumns:", list(df_bias.columns))

required = {"stored_needle", "layer", "requested_distance", "actual_eviction_distance", "filler_idx"}
missing = required - set(df_bias.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

prompt_keys = ["stored_needle", "requested_distance", "actual_eviction_distance", "filler_idx"]
prompt_keys += [c for c in ["needle_pos", "n_tokens", "prompt_tokens", "prompt"] if c in df_bias.columns]
prompts = df_bias[prompt_keys].drop_duplicates().reset_index(drop=True)
print("\nUnique prompt-level rows:", len(prompts))

_STATS = dict(n="count", mean="mean", std="std", min="min", median="median", max="max")

# A / B — balance across stored needles
def _balance(field, title):
    print("\n" + "=" * 72 + f"\n{title}\n" + "=" * 72)
    tab = pd.crosstab(prompts[field], prompts["stored_needle"]); print(tab)
    ok = (tab.nunique(axis=1) == 1).all()
    print("\nBalanced across stored needles:", "PASS" if ok else "CHECK")
    return ok
balanced_requested = _balance("requested_distance", "A. REQUESTED DISTANCE × STORED NEEDLE")
balanced_filler    = _balance("filler_idx", "B. FILLER × STORED NEEDLE")

# C — actual eviction distance
print("\n" + "=" * 72 + "\nC. ACTUAL EVICTION DISTANCE BY STORED NEEDLE\n" + "=" * 72)
print(prompts.groupby("stored_needle")["actual_eviction_distance"].agg(**_STATS))
print("\nMean actual eviction distance by requested distance:")
ev = prompts.groupby(["requested_distance", "stored_needle"])["actual_eviction_distance"].mean().unstack()
print(ev)
ev_spread = ev.max(axis=1) - ev.min(axis=1)
print("\nMax needle-to-needle spread within each requested distance:"); print(ev_spread)
print("\nLargest spread:", ev_spread.max())

# D — needle position
print("\n" + "=" * 72 + "\nD. NEEDLE POSITION\n" + "=" * 72)
if "needle_pos" in df_bias.columns:
    pp = df_bias[["stored_needle", "requested_distance", "filler_idx", "needle_pos"]].drop_duplicates()
    print(pp.groupby("stored_needle")["needle_pos"].agg(**_STATS))
    piv = pp.pivot_table(index=["requested_distance", "filler_idx"], columns="stored_needle",
                         values="needle_pos", aggfunc="mean")
    print("\nLargest within-condition needle-position spread:", (piv.max(axis=1) - piv.min(axis=1)).max())
else:
    print("NOT AVAILABLE in df_bias.\nCannot claim needle-position leakage has been ruled out.")

# E — prompt token length
print("\n" + "=" * 72 + "\nE. PROMPT TOKEN LENGTH\n" + "=" * 72)
tlc = next((c for c in ["n_tokens", "prompt_n_tokens", "prompt_length", "token_length"]
            if c in df_bias.columns), None)
if tlc:
    lp = df_bias[["stored_needle", "requested_distance", "filler_idx", tlc]].drop_duplicates()
    print(lp.groupby("stored_needle")[tlc].agg(**_STATS))
    piv = lp.pivot_table(index=["requested_distance", "filler_idx"], columns="stored_needle",
                         values=tlc, aggfunc="mean")
    print("\nLargest within-condition token-length spread:", (piv.max(axis=1) - piv.min(axis=1)).max())
else:
    print("NOT AVAILABLE in df_bias.\nCannot claim prompt-token-length leakage has been ruled out.")

# F — local context
print("\n" + "=" * 72 + "\nF. LOCAL CONTEXT AROUND NEEDLE\n" + "=" * 72)
context_cols = [c for c in ["prompt", "local_context", "needle_context", "context"] if c in df_bias.columns]
if context_cols:
    print("Available context columns:", context_cols)
    for c in context_cols:
        print(f"\nUnique {c} values:", df_bias[c].nunique())
    print("\nContext text is available. Inspect matched windows before declaring passed.")
else:
    print("NOT AVAILABLE in df_bias.\nLocal filler/context equivalence cannot be verified.")

# FINAL STATUS
print("\n" + "=" * 72 + "\nLEAKAGE CHECK STATUS\n" + "=" * 72)
print("Requested-distance balance:", "PASS" if balanced_requested else "CHECK")
print("Filler balance:", "PASS" if balanced_filler else "CHECK")
print("Actual eviction distance: INSPECT TABLE ABOVE")
print("Needle position:", "AVAILABLE" if "needle_pos" in df_bias.columns else "NOT AVAILABLE")
print("Prompt token length:", "AVAILABLE" if tlc else "NOT AVAILABLE")
print("Local prompt context:", "AVAILABLE" if context_cols else "NOT AVAILABLE")
print("\nDONE.")

In [ ]:
# C2 TOKEN-LENGTH / NEEDLE-POSITION CHECK (CPU only)
print("=" * 72 + "\nC2 TOKEN-LENGTH / NEEDLE-POSITION CHECK\n" + "=" * 72)
C2_NEEDLES = ["Paris", "Tokyo", "banana", "lantern"]

# c2 still has the original per-prompt metadata
meta = (c2[c2["needle"].isin(C2_NEEDLES)]
        [["needle", "requested_distance", "filler_idx", "n_tokens", "needle_pos", "eviction_distance"]]
        .drop_duplicates().copy())
print("\nUnique metadata rows:", len(meta))
print("\nRows per needle:"); print(meta.groupby("needle").size())

_AGG = ["count", "mean", "std", "min", "median", "max"]
print("\n" + "=" * 72 + "\n1. TOKEN LENGTH BY NEEDLE\n" + "=" * 72)
print(meta.groupby("needle")["n_tokens"].agg(_AGG))
print("\n" + "=" * 72 + "\n2. NEEDLE POSITION BY NEEDLE\n" + "=" * 72)
print(meta.groupby("needle")["needle_pos"].agg(_AGG))

# within-condition spread of each metadata field across the 4 needles
def spread(col):
    piv = meta.pivot_table(index=["requested_distance", "filler_idx"], columns="needle",
                           values=col, aggfunc="mean")
    return piv.max(axis=1) - piv.min(axis=1)
length_spread, pos_spread, evict_spread = spread("n_tokens"), spread("needle_pos"), spread("eviction_distance")

print("\n" + "=" * 72 + "\nWITHIN-CONDITION SPREADS\n" + "=" * 72)
print("\nToken-length spread:");     print(length_spread)
print("\nNeedle-position spread:");  print(pos_spread)
print("\nEviction-distance spread:"); print(evict_spread)

p

In [ ]:
# C2 #6 — SCRAMBLED-NEEDLE CONTENT CONTROL. Mirrors the bias sweep but STORES a neutral
# control word while reading out the ORIGINAL needle/distractor pair. Tests content-drift.
LAYERS = EXP["layers"]
DISTANCES = EXP["eviction_distances"]
FILLERS = range(EXP["n_filler_variants"])
PAIRS = {"Paris": "London", "Tokyo": "Osaka", "banana": "mango", "lantern": "torch"}
CONTROL_POOL = ["river","chair","window","garden","table","house","water","paper","stone","music",
                "green","cloud","flower","coffee","bridge","forest","silver","camera","bottle","pencil",
                "yellow","summer","winter","kitchen","street","book","door","tree"]

# 1. VERIFY PAIRS + CHOOSE CONTROLS
pair_ids = {n: (encode1(n), encode1(d)) for n, d in PAIRS.items()}
forbidden = set(PAIRS) | set(PAIRS.values())
valid_controls = [w for w in CONTROL_POOL
                  if len(tok.encode(f" {w}", add_special_tokens=False)) == 1 and w not in forbidden]
assert len(valid_controls) >= len(PAIRS), "Not enough verified single-token control words."
CONTROL_MAP = {n: valid_controls[i] for i, n in enumerate(PAIRS)}

print("=" * 72 + "\nSCRAMBLED-NEEDLE CONTROL MAP\n" + "=" * 72)
for n, ctrl in CONTROL_MAP.items():
    nid, cid = encode1(n), encode1(ctrl)
    print(f"{n:8s} -> {ctrl:8s} | [{nid}] -> [{cid}]")
    assert nid != cid

# 2. VERIFY df_bias
required_cols = {"stored_needle","tested_needle","distractor","layer","requested_distance",
                 "actual_eviction_distance","filler_idx","p_needle","p_distractor"}
missing = required_cols - set(df_bias.columns)
assert not missing, f"df_bias missing columns: {missing}"
original_target = df_bias[df_bias["stored_needle"] == df_bias["tested_needle"]].copy()
assert len(original_target) == 252, f"Expected 252 original target rows, found {len(original_target)}"
assert (original_target.groupby(["stored_needle","layer","requested_distance","filler_idx"]).size() == 1).all()

# 3. PRE-FLIGHT STRUCTURAL CHECK — replacing the needle must not change structure. No probe.run yet.
preflight_rows = []
for on_, cn_ in CONTROL_MAP.items():
    for dist in DISTANCES:
        for f in FILLERS:
            os_ = ai.build_niah_prompt(tok, on_, bundle, eviction_distance=dist, in_window=False, filler_idx=f)
            cs_ = ai.build_niah_prompt(tok, cn_, bundle, eviction_distance=dist, in_window=False, filler_idx=f)
            preflight_rows.append({
                "original_needle": on_, "control_needle": cn_, "requested_distance": dist, "filler_idx": f,
                "token_diff": cs_["n_tokens"] - os_["n_tokens"],
                "position_diff": cs_["needle_pos"] - os_["needle_pos"],
                "eviction_diff": cs_["actual_eviction_distance"] - os_["actual_eviction_distance"],
                "original_ahn_active": os_["ahn_will_activate"], "control_ahn_active": cs_["ahn_will_activate"],
                "original_evicted": os_["needle_is_evicted"], "control_evicted": cs_["needle_is_evicted"],
            })
df_content_preflight = pd.DataFrame(preflight_rows)
assert len(df_content_preflight) == 84
print("\n" + "=" * 72 + "\nSTRUCTURAL PREFLIGHT\n" + "=" * 72)
print("Max |token-count difference|:", df_content_preflight["token_diff"].abs().max())
print("Max |needle-position difference|:", df_content_preflight["position_diff"].abs().max())
print("Max |eviction-distance difference|:", df_content_preflight["eviction_diff"].abs().max())
assert (df_content_preflight["token_diff"] == 0).all(),    "STOP: token counts differ."
assert (df_content_preflight["position_diff"] == 0).all(), "STOP: needle positions differ."
assert (df_content_preflight["eviction_diff"] == 0).all(), "STOP: eviction distances differ."
assert (df_content_preflight["original_ahn_active"] == df_content_preflight["control_ahn_active"]).all(), "STOP: AHN activation differs."
assert (df_content_preflight["original_evicted"] == df_content_preflight["control_evicted"]).all(), "STOP: eviction status differs."
assert df_content_preflight["control_ahn_active"].all(), "STOP: a control prompt does not activate AHN."
assert df_content_preflight["control_evicted"].all(),    "STOP: a control needle is not evicted."
print("Structural preflight: PASS")

# 4. CONTROL SWEEP — store control_needle, read out the ORIGINAL needle/distractor pair (via score_word)
c2_content_control_rows = []
total = len(CONTROL_MAP) * len(DISTANCES) * len(list(FILLERS)); done = 0
for on_, cn_ in CONTROL_MAP.items():
    nid, did = pair_ids[on_]
    distractor = PAIRS[on_]
    for dist in DISTANCES:
        for f in FILLERS:
            scores, spec = score_word(cn_, {"needle": nid, "distractor": did}, dist, f)
            assert spec["ahn_will_activate"] and spec["needle_is_evicted"]
            if scores is None:
                continue
            for L, s in scores.items():
                (_, p_n) = s["needle"]; (_, p_d) = s["distractor"]
                assert p_n > 0 and p_d > 0
                c2_content_control_rows.append({
                    "original_needle": on_, "control_needle": cn_, "tested_needle": on_,
                    "distractor": distractor, "layer": L, "requested_distance": dist,
                    "actual_eviction_distance": spec["actual_eviction_distance"], "filler_idx": f,
                    "n_tokens": spec["n_tokens"], "needle_pos": spec["needle_pos"],
                    "p_needle": p_n, "p_distractor": p_d, "log_ratio": np.log(p_n) - np.log(p_d),
                })
            done += 1
            if done % 10 == 0 or done == total:
                print(f"{done}/{total} forward passes completed")
df_content_control = pd.DataFrame(c2_content_control_rows)

# 5. POST-RUN INTEGRITY
assert len(df_content_control) == 252, f"Expected 252 control rows, found {len(df_content_control)}"
assert (df_content_control.groupby(["original_needle","layer","requested_distance","filler_idx"]).size() == 1).all()
assert (original_target["p_needle"] > 0).all() and (original_target["p_distractor"] > 0).all()
original_target["original_log_ratio"] = np.log(original_target["p_needle"]) - np.log(original_target["p_distractor"])

original_for_merge = original_target[["stored_needle","layer","requested_distance","filler_idx",
                                      "actual_eviction_distance","original_log_ratio"]].rename(
    columns={"stored_needle": "original_needle", "actual_eviction_distance": "original_eviction_distance"})
paired_content = original_for_merge.merge(
    df_content_control[["original_needle","control_needle","layer","requested_distance","filler_idx",
                        "actual_eviction_distance","log_ratio"]].rename(
        columns={"actual_eviction_distance": "control_eviction_distance", "log_ratio": "control_log_ratio"}),
    on=["original_needle","layer","requested_distance","filler_idx"], how="inner", validate="one_to_one")
assert len(paired_content) == 252
assert (paired_content["original_eviction_distance"] == paired_content["control_eviction_distance"]).all()

# 6. ORIGINAL vs SCRAMBLED-CONTENT EFFECT (positive = stored word raises its own pair readout)
paired_content["delta_log_original_vs_control"] = (paired_content["original_log_ratio"]
                                                   - paired_content["control_log_ratio"])
paired_content["fold_original_vs_control"] = np.exp(paired_content["delta_log_original_vs_control"])

# 7. CONDITION-LEVEL (average the 4 pairs within each layer×distance×filler)
content_cond = (paired_content.groupby(["layer","requested_distance","filler_idx"], as_index=False)
                .agg(delta_log=("delta_log_original_vs_control","mean"),
                     n_pairs=("delta_log_original_vs_control","size")))
assert (content_cond["n_pairs"] == 4).all()
assert len(content_cond) == 63

# 8. REPORT
print("\n" + "=" * 72 + "\nC2 #6 SCRAMBLED-NEEDLE CONTENT CONTROL\n" + "=" * 72)
print("\nControl rows:", len(df_content_control), "| Paired rows:", len(paired_content),
      "| Condition-level rows:", len(content_cond))
_ff = lambda x: f"{x:.6g}"
content_summary_rows = []
for L, g in content_cond.groupby("layer"):
    x = g["delta_log"].to_numpy(dtype=float); n = len(x)
    assert n == 21
    mean_log = x.mean(); se = x.std(ddof=1) / np.sqrt(n); tcrit = stats.t.ppf(0.975, df=n - 1)
    t_stat, p_value = stats.ttest_1samp(x, popmean=0.0)
    content_summary_rows.append({"layer": L, "n_conditions": n, "mean_log_effect": mean_log,
                                 "geom_fold": np.exp(mean_log), "CI_low": np.exp(mean_log - tcrit * se),
                                 "CI_high": np.exp(mean_log + tcrit * se), "t": t_stat, "p": p_value})
content_summary = pd.DataFrame(content_summary_rows)
print("\n"); print(content_summary.to_string(index=False, float_format=_ff))

# 9. PER-NEEDLE
print("\n" + "=" * 72 + "\nPER-NEEDLE CONTENT CONTROL\n" + "=" * 72)
per_needle_content = (paired_content.groupby(["layer","original_needle","control_needle"])
                      .agg(n=("delta_log_original_vs_control","size"),
                           mean_log=("delta_log_original_vs_control","mean")).reset_index())
per_needle_content["geom_fold"] = np.exp(per_needle_content["mean_log"])
print(per_needle_content.to_string(index=False, float_format=_ff))
print("\nDONE.")

# 04 — NIAH C2 Follow-up: Final Summary

**Checkpoint:** Qwen2.5-3B-Instruct + AHN-GDN  
**Readout:** J-Lens (Table 3 not passed — all findings conditional)  
**Layers:** 9, 18, 27

---

## Main result

C2 does not provide convincing memory-specific evidence on this checkpoint.

The original C2 >=10× control fails, and raw needle/distractor ratios are strongly affected by pair-specific readout preferences.

After correcting for pair-specific baseline preference and using 21 condition-level observations per layer:

| Layer | Fold | 95% CI | p |
|---:|---:|:---:|---:|
| 9 | 1.000× | [0.989, 1.012] | 0.948 |
| 18 | 1.045× | [1.019, 1.072] | 0.00183 |
| 27 | 0.990× | [0.965, 1.017] | 0.455 |

Layers 9 and 27 are null.

Layer 18 shows a small positive association in the pair-baseline-corrected diagnostic and remains stable across leave-one-distance-out, leave-one-filler-out, and blocked-permutation checks.

However, the scrambled-needle content control does not preserve that Layer-18 effect:

| Layer | Fold | 95% CI | p |
|---:|---:|:---:|---:|
| 9 | 0.960× | [0.936, 0.984] | 0.00260 |
| 18 | 1.012× | [0.980, 1.046] | 0.441 |
| 27 | 0.999× | [0.952, 1.048] | 0.958 |

Therefore, the Layer-18 positive association cannot currently be treated as a memory-specific effect.

## Conclusion

- C2 fails its original >=10× criterion.
- Layer 9 shows no positive aggregate C2 effect.
- Layer 18 shows a small positive corrected association, but it does not survive the scrambled-content control.
- Layer 27 shows no aggregate corrected effect with the actual J-Lens loaded.
- No tested layer currently provides convincing positive, retention-consistent, memory-specific evidence.

This does not imply that AHN retains no information. It only means that the current C2 + J-Lens readout does not provide a validated token-level memory signal on this checkpoint.

All conclusions remain conditional because the J-Lens has not passed the full Table 3 validation battery.

# C2/C3 Pipeline Debugging

## Goal

Trace the existing C2 and C3 controls through the same five links requested by Gautam:

1. Input
2. Eviction
3. AHN output (`o_t`)
4. J-Lens readout
5. Final control metric

The goal is **not to make C2 or C3 pass**. The goal is to identify the first stage where the observed behavior stops matching the control design.

Possible diagnoses include:

- input or eviction mismatch → implementation issue
- control changes something unintended → control-design issue
- AHN states do not differ as expected → AHN-state issue
- AHN states differ but J-Lens does not preserve the difference → J-Lens/readout issue
- all links behave as implemented but the control still fails → genuine negative result or incorrect control expectation

## Debugging strategy

Start with one small representative case and inspect each link before moving to the next. Avoid full reruns unless the small trace shows they are necessary.

Current C2 diagnostic case:
- target: `Paris`
- content-control token: `river`
- readout pair: `Paris` vs `London`
- distance: `1024`
- filler: `0`
- layers: `9, 18, 27`

Each link will be marked **PASS**, **FAIL**, or **NEEDS CHECKING** before continuing.

In [ ]:
def c2_input_trace(original, control, distractor, dist, filler, k=6):
    """C2 input equivalence, one case. CPU only, no forward pass.
    Confirms real vs control prompts differ by exactly one token."""
    oid = tok.encode(f" {original}",   add_special_tokens=False); assert len(oid) == 1
    cid = tok.encode(f" {control}",    add_special_tokens=False); assert len(cid) == 1
    did = tok.encode(f" {distractor}", add_special_tokens=False); assert len(did) == 1
    oid, cid, did = oid[0], cid[0], did[0]
    sr = ai.build_niah_prompt(tok, original, bundle, eviction_distance=dist, in_window=False, filler_idx=filler)
    sc = ai.build_niah_prompt(tok, control,  bundle, eviction_distance=dist, in_window=False, filler_idx=filler)
    ir = tok(sr["prompt"], return_tensors="pt")["input_ids"][0].tolist()
    ic = tok(sc["prompt"], return_tensors="pt")["input_ids"][0].tolist()
    print(f"case: {original!r} vs {control!r}, pair readout {original}/{distractor}, dist={dist}, filler={filler}\n")
    print(f"token ids: {original}={oid}  {control}={cid}  {distractor}={did}\n")
    print(f"spec['n_tokens']    : real={sr['n_tokens']}  ctrl={sc['n_tokens']}")
    print(f"spec['needle_pos']  : real={sr['needle_pos']}  ctrl={sc['needle_pos']}")
    print(f"tokenized length    : real={len(ir)}  ctrl={len(ic)}\n")
    diffs = [i for i in range(min(len(ir), len(ic))) if ir[i] != ic[i]]
    print(f"positions differing between real and ctrl tokenized inputs: {len(diffs)}")
    print(f"differing positions: {diffs[:20]}{' ...' if len(diffs) > 20 else ''}")
    for label, ids, pos in [("real", ir, sr["needle_pos"]), ("ctrl", ic, sc["needle_pos"])]:
        lo, hi = max(0, pos - k), min(len(ids), pos + k + 1)
        print(f"\n{label}  tokens around spec needle_pos={pos}  (positions {lo}..{hi-1}):")
        for i in range(lo, hi):
            marker = "  <-- needle_pos" if i == pos else ""
            print(f"  [{i}] id={ids[i]:>7d}  {tok.decode([ids[i]])!r}{marker}")
    if len(diffs) == 1:
        p = diffs[0]
        print(f"\ndiff at position {p}:")
        print(f"  real id={ir[p]}  {tok.decode([ir[p]])!r}")
        print(f"  ctrl id={ic[p]}  {tok.decode([ic[p]])!r}")
    return dict(spec_real=sr, spec_ctrl=sc, diffs=diffs)

# Stage 1 — Paris vs river
_ = c2_input_trace("Paris", "river", "London", dist=1024, filler=0)

### Link 1 — Input equivalence: PASS

For the C2 diagnostic case, the real prompt stores `Paris` and the control prompt stores `river`.

The two tokenized inputs have:
- the same total token count
- the same needle position
- exactly one differing token
- that difference is only `Paris` vs `river`

After correcting `needle_pos`, both conditions now correctly report the actual stored token at position 148.

This confirms that the C2 control inputs are structurally matched as intended, so the observed C2 behavior is not caused by an input-construction mismatch.

### Link 2 — Eviction equivalence: PASS

For the diagnostic Paris vs river case, both prompts have the same compression boundary (1188), the same actual eviction distance (1040), and both stored tokens are evicted. This means the control is not failing because one condition is being retained in the local window while the other is compressed.

So far:
- Link 1 Input: PASS
- Link 2 Eviction: PASS

The first possible failure point is now downstream, starting with the AHN output tensor (`o_t`).

In [ ]:
r = c2_input_trace("Paris", "river", "London", dist=1024, filler=0)
ins_real = tok(r["spec_real"]["prompt"], return_tensors="pt").to(bundle.model.device)
ins_ctrl = tok(r["spec_ctrl"]["prompt"], return_tensors="pt").to(bundle.model.device)
on_real = probe.run(ins_real, nowrite=False, layers=EXP["layers"], capture_residual=True)
on_ctrl = probe.run(ins_ctrl, nowrite=False, layers=EXP["layers"], capture_residual=True)

In [ ]:
for L in EXP["layers"]:
    o_real = on_real.o_t(L, pos=-1).float()
    o_ctrl = on_ctrl.o_t(L, pos=-1).float()
    diff = o_real - o_ctrl
    cos = F.cosine_similarity(o_real.flatten(), o_ctrl.flatten(), dim=0).item()
    print(f"\nLayer {L}")
    print("||o_t_real||:", o_real.norm().item())
    print("||o_t_ctrl||:", o_ctrl.norm().item())
    print("||diff||:", diff.norm().item())
    print("relative_diff:", diff.norm().item() / max(o_real.norm().item(), 1e-12))
    print("cosine:", cos)

### Link 3 — AHN output comparison: DIFFERENCE PRESENT

The target-stored (`Paris`) and control-stored (`river`) conditions do not produce identical AHN `o_t` states.

- Layer 9: relative difference ≈ 2.5%, cosine ≈ 0.9997
- Layer 18: relative difference ≈ 6.9%, cosine ≈ 0.9977
- Layer 27: relative difference ≈ 8.0%, cosine ≈ 0.9970

The states remain highly aligned overall, but the difference grows at deeper layers. Therefore the C2 control does produce a measurable token-specific change in the AHN state.

This does not yet tell us whether that difference is memory-specific or whether J-Lens preserves it. The next step is Link 4: compare the J-Lens readouts for Paris and London from these same tensors.

In [ ]:
tokenizer = bundle.tokenizer

In [ ]:
paris_id, london_id = encode1("Paris"), encode1("London")
for L in EXP["layers"]:
    lg_real = ai.readout_logits(on_real.o_t(L, pos=-1), bundle, lens=lens if EXP["use_jlens"] else None, layer=L)
    lg_ctrl = ai.readout_logits(on_ctrl.o_t(L, pos=-1), bundle, lens=lens if EXP["use_jlens"] else None, layer=L)
    p_real = torch.softmax(lg_real.float(), dim=-1)
    p_ctrl = torch.softmax(lg_ctrl.float(), dim=-1)
    real_lr = torch.log(p_real[paris_id]) - torch.log(p_real[london_id])
    ctrl_lr = torch.log(p_ctrl[paris_id]) - torch.log(p_ctrl[london_id])
    print(f"\nLayer {L}")
    print("real  p(Paris): ", p_real[paris_id].item())
    print("real  p(London):", p_real[london_id].item())
    print("ctrl  p(Paris): ", p_ctrl[paris_id].item())
    print("ctrl  p(London):", p_ctrl[london_id].item())
    print("real log-ratio:", real_lr.item())
    print("ctrl log-ratio:", ctrl_lr.item())
    print("delta:", (real_lr - ctrl_lr).item())
    print("real rank Paris:", ai.token_rank(lg_real, paris_id))
    print("ctrl rank Paris:", ai.token_rank(lg_ctrl, paris_id))

### Link 4 — J-Lens readout: DIFFERENCE PRESERVED

The J-Lens readout preserves some of the difference between the `Paris`-stored and `river`-stored AHN states.

- Layer 9: delta log-ratio = -0.021
- Layer 18: delta log-ratio = +0.063
- Layer 27: delta log-ratio = +0.295

At Layers 18 and 27, storing `Paris` increases the Paris-vs-London readout relative to storing `river`.

Therefore, for this diagnostic case, the token-specific difference present in `o_t` is not completely lost by J-Lens. The next step is Link 5: verify that this per-case delta matches the existing C2 control row used in the aggregate analysis.

In [ ]:
check = paired_content[
    (paired_content["original_needle"] == "Paris")
    & (paired_content["requested_distance"] == 1024)
    & (paired_content["filler_idx"] == 0)
][["layer", "original_log_ratio", "control_log_ratio", "delta_log_original_vs_control",
   "fold_original_vs_control", "original_eviction_distance", "control_eviction_distance"]]
print(check.to_string(index=False))

### Link 5 — Control metric cross-check: PASS

For the diagnostic case (`Paris` stored vs `river` stored, distance 1024, filler 0), the per-layer deltas recomputed directly from the J-Lens readouts exactly match the corresponding rows already stored in `paired_content`.

- Layer 9: -0.021412
- Layer 18: +0.062548
- Layer 27: +0.294891

This confirms that the C2 summary bookkeeping is faithful to the underlying readout for this case. The unexpected aggregate C2 result is therefore not caused by a mismatch between the live readout and the saved control metric for this diagnostic example.

In [ ]:
summary = (
    paired_content
    .groupby(["original_needle", "layer"])["delta_log_original_vs_control"]
    .agg(["mean", "std", "min", "max", "count"])
    .reset_index()
)
print(summary.to_string(index=False))

### C2 pair-level heterogeneity

The C2 control effect is not consistent across stored needles.

At Layer 18:
- Paris: mean delta = +0.033
- Tokyo: mean delta = +0.002
- banana: mean delta = +0.053
- lantern: mean delta = -0.051

At Layer 27 the disagreement is stronger:
- Paris and Tokyo are positive
- banana and lantern are negative

The within-needle ranges are also large, including both positive and negative conditions.

Therefore, the weak aggregate C2 result is not explained by a single broken summary calculation. The control effect varies substantially by needle and experimental condition, suggesting content/pair dependence or instability in the J-Lens-derived signal.

In [ ]:
worst_lantern = (paired_content[(paired_content["original_needle"] == "lantern")
                                & (paired_content["layer"] == 18)]
                 .sort_values("delta_log_original_vs_control").head(5))
print(worst_lantern[["requested_distance", "filler_idx", "original_log_ratio", "control_log_ratio",
                     "delta_log_original_vs_control", "fold_original_vs_control",
                     "original_eviction_distance", "control_eviction_distance"]].to_string(index=False))

In [ ]:
# Stage 1 — lantern vs garden, pair readout lantern/torch
r = c2_input_trace("lantern", "garden", "torch", dist=64, filler=2)
spec_real, spec_ctrl = r["spec_real"], r["spec_ctrl"]
ins_real = tok(spec_real["prompt"], return_tensors="pt").to(bundle.model.device)
ins_ctrl = tok(spec_ctrl["prompt"], return_tensors="pt").to(bundle.model.device)
on_real = probe.run(ins_real, nowrite=False, layers=EXP["layers"], capture_residual=True)
on_ctrl = probe.run(ins_ctrl, nowrite=False, layers=EXP["layers"], capture_residual=True)

In [ ]:
for L in EXP["layers"]:
    o_real = on_real.o_t(L, pos=-1).float()
    o_ctrl = on_ctrl.o_t(L, pos=-1).float()
    diff = o_real - o_ctrl

    print(f"\nLayer {L}")
    print("||o_t_real||:", o_real.norm().item())
    print("||o_t_ctrl||:", o_ctrl.norm().item())
    print("||diff||:", diff.norm().item())
    print("relative_diff:", diff.norm().item() / max(o_real.norm().item(), 1e-12))
    print("cosine:", F.cosine_similarity(o_real.flatten(), o_ctrl.flatten(), dim=0).item())

In [ ]:
needle_id, distractor_id = encode1("lantern"), encode1("torch")
print("readout pair: lantern", needle_id, "/ torch", distractor_id)
for L in EXP["layers"]:
    lg_real = ai.readout_logits(on_real.o_t(L, pos=-1), bundle, lens=lens if EXP["use_jlens"] else None, layer=L)
    lg_ctrl = ai.readout_logits(on_ctrl.o_t(L, pos=-1), bundle, lens=lens if EXP["use_jlens"] else None, layer=L)
    p_real = torch.softmax(lg_real.float(), dim=-1)
    p_ctrl = torch.softmax(lg_ctrl.float(), dim=-1)
    real_lr = torch.log(p_real[needle_id]) - torch.log(p_real[distractor_id])
    ctrl_lr = torch.log(p_ctrl[needle_id]) - torch.log(p_ctrl[distractor_id])
    print(f"\nLayer {L}")
    print("real p(lantern):", p_real[needle_id].item())
    print("real p(torch):  ", p_real[distractor_id].item())
    print("ctrl p(lantern):", p_ctrl[needle_id].item())
    print("ctrl p(torch):  ", p_ctrl[distractor_id].item())
    print("real log-ratio:", real_lr.item())
    print("ctrl log-ratio:", ctrl_lr.item())
    print("delta:", (real_lr - ctrl_lr).item())
    print("real rank lantern:", ai.token_rank(lg_real, needle_id))
    print("ctrl rank lantern:", ai.token_rank(lg_ctrl, needle_id))

In [ ]:
check_lantern = paired_content[
    (paired_content["original_needle"] == "lantern")
    & (paired_content["requested_distance"] == 64)
    & (paired_content["filler_idx"] == 2)
][["layer", "original_log_ratio", "control_log_ratio", "delta_log_original_vs_control",
   "fold_original_vs_control", "original_eviction_distance", "control_eviction_distance"]]
print(check_lantern.to_string(index=False))

### C2 five-link trace — negative lantern case

For `lantern` vs unrelated control `garden`
(requested distance 64, filler 2; readout pair `lantern/torch`):

1. **Input: PASS**
   - same sequence length
   - same needle position
   - exactly one token differs (`lantern` vs `garden`)

2. **Eviction: PASS**
   - same compression boundary
   - same actual eviction distance = 84
   - both stored words are evicted

3. **AHN output: DIFFERENCE PRESENT**
   - Layer 18 relative `o_t` difference ≈ 11.7%
   - cosine ≈ 0.9933
   - therefore the two conditions are distinguishable in AHN state space

4. **J-Lens readout: UNEXPECTED RELATIVE DIRECTION**
   - Layer 18 real log-ratio (`lantern/torch`) = 2.750920
   - control log-ratio = 3.800121
   - delta = -1.049201
   - the control state therefore shows a stronger relative `lantern/torch`
     preference than the actual lantern-stored state

5. **Control metric: PASS**
   - the live recomputation exactly matches `paired_content`
   - therefore the negative C2 value is not caused by summary/bookkeeping error

For this diagnostic case, the pipeline is structurally consistent through the
final metric. The unexpected C2 behavior appears downstream of a genuine AHN
state difference, in the relationship between that state difference and the
J-Lens target-vs-distractor readout.

In [ ]:
needle_id, distractor_id = encode1("lantern"), encode1("torch")
print("pair: lantern / torch")
for L in EXP["layers"]:
    o_real, o_ctrl = on_real.o_t(L, pos=-1), on_ctrl.o_t(L, pos=-1)
    j_real  = ai.readout_logits(o_real, bundle, lens=lens, layer=L)
    j_ctrl  = ai.readout_logits(o_ctrl, bundle, lens=lens, layer=L)
    plain_real = ai.readout_logits(o_real, bundle, lens=None)
    plain_ctrl = ai.readout_logits(o_ctrl, bundle, lens=None)
    j_delta     = (j_real[needle_id] - j_real[distractor_id]) - (j_ctrl[needle_id] - j_ctrl[distractor_id])
    plain_delta = (plain_real[needle_id] - plain_real[distractor_id]) - (plain_ctrl[needle_id] - plain_ctrl[distractor_id])
    print(f"\nLayer {L}")
    print("J-Lens delta:", j_delta.item())
    print("Plain delta: ", plain_delta.item())
    print("J real rank needle:",     ai.token_rank(j_real, needle_id))
    print("J ctrl rank needle:",     ai.token_rank(j_ctrl, needle_id))
    print("Plain real rank needle:", ai.token_rank(plain_real, needle_id))
    print("Plain ctrl rank needle:", ai.token_rank(plain_ctrl, needle_id))

### Plain-readout vs J-Lens diagnostic

For the strongest negative C2 case (`lantern` vs `garden`,
readout pair `lantern/torch`), the negative corrected effect is present
under both readout methods.

Layer 18:
- J-Lens delta = -1.0492
- Plain readout delta = -0.6560

Therefore, the negative C2 direction is not created solely by the J-Lens
transformation. The same direction is already present when the AHN `o_t`
state is decoded with the plain vocabulary readout.

This shifts the likely explanation away from a J-Lens-specific sign reversal
and toward either content-dependent AHN state geometry or the target-vs-
distractor control metric itself.

In [ ]:
L = 18
lg_real = ai.readout_logits(on_real.o_t(L, pos=-1), bundle, lens=lens, layer=L)
lg_ctrl = ai.readout_logits(on_ctrl.o_t(L, pos=-1), bundle, lens=lens, layer=L)
lantern_real, lantern_ctrl = lg_real[needle_id].item(), lg_ctrl[needle_id].item()
torch_real, torch_ctrl     = lg_real[distractor_id].item(), lg_ctrl[distractor_id].item()
print("Layer 18")
print("lantern real logit:", lantern_real)
print("lantern ctrl logit:", lantern_ctrl)
print("lantern change real-ctrl:", lantern_real - lantern_ctrl)
print("\ntorch real logit:", torch_real)
print("torch ctrl logit:", torch_ctrl)
print("torch change real-ctrl:", torch_real - torch_ctrl)
print("\npair delta:", (lantern_real - torch_real) - (lantern_ctrl - torch_ctrl))

### Why the Layer-18 C2 delta is negative

For the `lantern` vs `garden` condition:

- `lantern` logit increases by +0.687
- `torch` logit increases by +1.736

Therefore the stored `lantern` does strengthen the target token, but it
strengthens the semantically related distractor `torch` even more.

This produces the negative corrected pair delta:

`(+0.687) - (+1.736) = -1.049`

So this C2 failure is not simply "the model does not retain lantern."
Instead, the target-vs-distractor metric is strongly affected by how the
stored content changes both members of the semantic pair.

In [ ]:
check = (paired_content[paired_content["layer"] == 18]
         .groupby("original_needle")
         .agg(mean_original_log_ratio=("original_log_ratio", "mean"),
              mean_control_log_ratio=("control_log_ratio", "mean"),
              mean_delta=("delta_log_original_vs_control", "mean"),
              min_delta=("delta_log_original_vs_control", "min"),
              max_delta=("delta_log_original_vs_control", "max"),
              n=("delta_log_original_vs_control", "size"))
         .reset_index())
print(check.to_string(index=False))

In [ ]:
best = (paired_content[paired_content["layer"] == 18]
        .sort_values("delta_log_original_vs_control", ascending=False).head(5))
print(best[["original_needle", "requested_distance", "filler_idx",
            "original_log_ratio", "control_log_ratio", "delta_log_original_vs_control"]]
      .to_string(index=False))

In [ ]:
# C2 positive comparison — lantern vs garden, dist 256, pair readout lantern/torch
DEBUG_ORIGINAL, DEBUG_CONTROL, DEBUG_DISTRACTOR, DEBUG_DIST, DEBUG_FILLER = "lantern", "garden", "torch", 256, 0
r = c2_input_trace(DEBUG_ORIGINAL, DEBUG_CONTROL, DEBUG_DISTRACTOR, dist=DEBUG_DIST, filler=DEBUG_FILLER)
spec_real, spec_ctrl = r["spec_real"], r["spec_ctrl"]
ins_real = tok(spec_real["prompt"], return_tensors="pt").to(bundle.model.device)
ins_ctrl = tok(spec_ctrl["prompt"], return_tensors="pt").to(bundle.model.device)
on_real = probe.run(ins_real, nowrite=False, layers=EXP["layers"], capture_residual=True)
on_ctrl = probe.run(ins_ctrl, nowrite=False, layers=EXP["layers"], capture_residual=True)
needle_id, distractor_id = encode1(DEBUG_ORIGINAL), encode1(DEBUG_DISTRACTOR)

In [ ]:
for L in EXP["layers"]:
    o_real = on_real.o_t(L, pos=-1).float()
    o_ctrl = on_ctrl.o_t(L, pos=-1).float()
    diff = o_real - o_ctrl
    cos = F.cosine_similarity(o_real.flatten(), o_ctrl.flatten(), dim=0).item()
    print(f"\nLayer {L}")
    print("||o_t_real||:", o_real.norm().item())
    print("||o_t_ctrl||:", o_ctrl.norm().item())
    print("||diff||:", diff.norm().item())
    print("relative_diff:", diff.norm().item() / max(o_real.norm().item(), 1e-12))
    print("cosine:", cos)

In [ ]:
L = 18
needle_id, distractor_id = encode1(DEBUG_ORIGINAL), encode1(DEBUG_DISTRACTOR)
lg_real = ai.readout_logits(on_real.o_t(L, pos=-1), bundle, lens=lens, layer=L)
lg_ctrl = ai.readout_logits(on_ctrl.o_t(L, pos=-1), bundle, lens=lens, layer=L)
needle_real, needle_ctrl = lg_real[needle_id].item(), lg_ctrl[needle_id].item()
dist_real, dist_ctrl     = lg_real[distractor_id].item(), lg_ctrl[distractor_id].item()
print("Layer 18")
print("target:", DEBUG_ORIGINAL, "| distractor:", DEBUG_DISTRACTOR)
print("\ntarget real logit:", needle_real)
print("target ctrl logit:", needle_ctrl)
print("target change:", needle_real - needle_ctrl)
print("\ndistractor real logit:", dist_real)
print("distractor ctrl logit:", dist_ctrl)
print("distractor change:", dist_real - dist_ctrl)
print("\npair delta:", (needle_real - needle_ctrl) - (dist_real - dist_ctrl))
print("real target rank:", ai.token_rank(lg_real, needle_id))
print("ctrl target rank:", ai.token_rank(lg_ctrl, needle_id))

### C2 debugging conclusion

The C2 pipeline was traced through input construction, eviction, AHN output,
J-Lens readout, and the final saved metric.

For both a strongly negative and a strongly positive `lantern/torch` condition:

- input construction is structurally matched
- eviction behavior is matched
- `o_t` differs between the real and content-control conditions
- the J-Lens readout reproduces the saved C2 metric
- the saved metric bookkeeping is correct

The difference between positive and negative C2 conditions comes from how the
target and distractor logits move relative to each other.

Strong negative condition:
- lantern logit change = +0.687
- torch logit change = +1.736
- pair delta = -1.049

Strong positive condition:
- lantern logit change = -0.189
- torch logit change = -0.660
- pair delta = +0.471

Therefore, the sign of the C2 statistic does not simply indicate whether the
stored needle became stronger in the AHN state. It depends on the relative
movement of the target and its semantic distractor.

For the cases inspected, no implementation or bookkeeping bug was found in the
five-link pipeline. The main issue appears to be that the target-vs-distractor
control metric is highly content- and condition-dependent.

This does not establish that J-Lens is valid or that AHN retains no information.
It shows that the current C2 statistic is not a clean memory-specific readout.

In [ ]:
# Original: target stored AND tested. Control: unrelated word stored, target still tested.
orig = (df_bias[df_bias["stored_needle"] == df_bias["tested_needle"]]
        [["stored_needle", "layer", "requested_distance", "filler_idx", "p_needle"]]
        .rename(columns={"stored_needle": "original_needle", "p_needle": "p_target_original"}))
ctrl = (df_content_control[df_content_control["original_needle"] == df_content_control["tested_needle"]]
        [["original_needle", "layer", "requested_distance", "filler_idx", "p_needle"]]
        .rename(columns={"p_needle": "p_target_control"}))
target_only = orig.merge(ctrl, on=["original_needle", "layer", "requested_distance", "filler_idx"],
                         validate="one_to_one")
# positive = target stronger when the target itself was stored
target_only["delta_log_target"] = (np.log(target_only["p_target_original"] + 1e-30)
                                   - np.log(target_only["p_target_control"] + 1e-30))
summary_target = (target_only.groupby(["original_needle", "layer"])["delta_log_target"]
                  .agg(["mean", "std", "min", "max", "count"]).reset_index())
print(summary_target.to_string(index=False))

In [ ]:
# Layer 18 first — where the corrected C2 effect appeared.
L = 18
rows = []
for target in ["Paris", "Tokyo", "banana", "lantern"]:
    actual = (df_bias[(df_bias["layer"] == L) & (df_bias["tested_needle"] == target)
                      & (df_bias["stored_needle"] == target)]
              [["requested_distance", "filler_idx", "p_needle"]]
              .rename(columns={"p_needle": "p_target_actual"}))
    others = (df_bias[(df_bias["layer"] == L) & (df_bias["tested_needle"] == target)
                      & (df_bias["stored_needle"] != target)]
              [["stored_needle", "requested_distance", "filler_idx", "p_needle"]]
              .rename(columns={"stored_needle": "control_stored", "p_needle": "p_target_other"}))
    merged = others.merge(actual, on=["requested_distance", "filler_idx"], validate="many_to_one")
    merged["target"] = target
    merged["delta_log_target"] = (np.log(merged["p_target_actual"] + 1e-30)
                                  - np.log(merged["p_target_other"] + 1e-30))
    rows.append(merged)
cross_control = pd.concat(rows, ignore_index=True)
summary = (cross_control.groupby(["target", "control_stored"])["delta_log_target"]
           .agg(["mean", "std", "min", "max", "count"]).reset_index())
print(summary.to_string(index=False))

In [ ]:
CONTROL_CANDIDATES = ["river", "chair", "window", "garden", "table", "house",
                      "water", "paper", "stone", "music", "green", "cloud"]
targets = ["Paris", "Tokyo", "banana", "lantern"]
valid_controls = [(w, tok.encode(f" {w}", add_special_tokens=False)[0])
                  for w in CONTROL_CANDIDATES
                  if len(tok.encode(f" {w}", add_special_tokens=False)) == 1 and w not in targets]
print("Valid single-token controls:")
for word, tid in valid_controls:
    print(f"{word:10s} -> {tid}")

In [ ]:
PILOT_CONTROLS = ["river", "chair", "window", "garden"]
TARGETS = ["Paris", "Tokyo", "banana", "lantern"]
problems = []
for target in TARGETS:
    for control in PILOT_CONTROLS:
        for distance in EXP["eviction_distances"]:
            for filler_idx in range(EXP["n_filler_variants"]):
                spec = ai.build_niah_prompt(tok, control, bundle, eviction_distance=distance,
                                            in_window=False, filler_idx=filler_idx)
                pl = spec["prompt"].lower()
                if target.lower() in pl:
                    problems.append((target, control, distance, filler_idx, "target leaked"))
                if pl.count(control.lower()) != 1:
                    problems.append((target, control, distance, filler_idx, f"control count={pl.count(control.lower())}"))
print("Problems found:", len(problems))
for row in problems[:20]:
    print(row)
if not problems:
    print("PASS: pilot control prompts are structurally clean.")

In [ ]:
# C2-v2 PILOT — multi-control target-only baseline. 84 forward passes; each scores all 4 targets.
PILOT_CONTROLS = ["river", "chair", "window", "garden"]
TARGETS = ["Paris", "Tokyo", "banana", "lantern"]
target_ids = {t: encode1(t) for t in TARGETS}

pilot_rows = []
total = len(PILOT_CONTROLS) * len(EXP["eviction_distances"]) * EXP["n_filler_variants"]
done = 0
for control in PILOT_CONTROLS:
    for distance in EXP["eviction_distances"]:
        for filler_idx in range(EXP["n_filler_variants"]):
            scores, spec = score_word(control, target_ids, distance, filler_idx)
            assert spec["needle_is_evicted"], (control, distance, filler_idx, spec["actual_eviction_distance"])
            if scores is None:
                continue
            for L, s in scores.items():
                for target in TARGETS:
                    pilot_rows.append({
                        "control_needle": control, "tested_target": target, "layer": L,
                        "requested_distance": distance,
                        "actual_eviction_distance": spec["actual_eviction_distance"],
                        "filler_idx": filler_idx, "p_target": s[target][1],  # (rank, prob) -> prob
                    })
            done += 1
            print(f"\r{done}/{total} forward passes", end="", flush=True)
print("\nDone.")
df_c2_multicontrol = pd.DataFrame(pilot_rows)
print("rows:", len(df_c2_multicontrol))
print(df_c2_multicontrol.head())

In [ ]:
# Original: target itself stored
orig = (df_bias[df_bias["stored_needle"] == df_bias["tested_needle"]]
        [["stored_needle", "layer", "requested_distance", "filler_idx", "p_needle"]]
        .rename(columns={"stored_needle": "target", "p_needle": "p_target_original"}))
# Multi-control baseline: mean log p(target) across the 4 unrelated controls
ctrl = df_c2_multicontrol.copy()
ctrl["log_p_target"] = np.log(ctrl["p_target"] + 1e-30)
ctrl_mean = (ctrl.groupby(["tested_target", "layer", "requested_distance", "filler_idx"])
             .agg(mean_log_p_control=("log_p_target", "mean"),
                  std_log_p_control=("log_p_target", "std"),
                  n_controls=("control_needle", "nunique"))
             .reset_index().rename(columns={"tested_target": "target"}))
# Match original to baseline
c2_v2 = orig.merge(ctrl_mean, on=["target", "layer", "requested_distance", "filler_idx"],
                   validate="one_to_one")
c2_v2["delta_log_target_multicontrol"] = (np.log(c2_v2["p_target_original"] + 1e-30)
                                          - c2_v2["mean_log_p_control"])
summary_c2_v2 = (c2_v2.groupby(["target", "layer"])["delta_log_target_multicontrol"]
                 .agg(["mean", "std", "min", "max", "count"]).reset_index())
print(summary_c2_v2.to_string(index=False))

## C2 Debugging Summary

We finished debugging **C2** through the full pipeline.

### What we checked

1. **Input**
   - Real and control prompts were structurally matched.
   - Only the stored word changed.

2. **Eviction**
   - Real and control conditions used the same compression boundary.
   - Both stored words were actually evicted at the same distance.

3. **AHN state (`o_t`)**
   - The AHN state changed between the real and control conditions.
   - So the model was not producing identical memory states.

4. **J-Lens readout**
   - J-Lens preserved differences between those AHN states.
   - The live J-Lens calculations matched the saved C2 results.

5. **Final metric**
   - The saved C2 calculations were correct.
   - We did not find a bookkeeping or aggregation bug.

### Why C2 is unstable

For some conditions, storing the correct target increased the target signal, but it also increased the semantic distractor even more.

Example at Layer 18 for `lantern/torch`:

- lantern logit change: `+0.687`
- torch logit change: `+1.736`
- resulting C2 delta: `-1.049`

So a negative C2 score does not necessarily mean the target was forgotten.

We also checked the same AHN states using the **plain vocabulary readout** instead of J-Lens. The negative direction was still present, so this is **not a J-Lens-only problem**.

### Attempts to improve C2

We tested:

- target-only comparison
- using other stored needles as controls
- a multi-control baseline using 4 unrelated control words

For the multi-control pilot we ran:

- **84 additional model forward passes**
- **1008 readout rows**
- Layers 9, 18, and 27

At Layer 18, the average multi-control effects were:

- Paris: `+0.049`
- Tokyo: `-0.026`
- banana: `+0.147`
- lantern: `+0.031`

The results were still mixed and highly variable.

### Current conclusion

We did **not** find evidence that C2 is failing because of:

- an input-construction bug
- an eviction mismatch
- a bookkeeping bug
- a J-Lens-only sign reversal

Instead, the current C2 readout appears to be **content-dependent and not consistently memory-specific**.

Even replacing the single control with multiple unrelated controls did not produce a stable positive signal.

### Status

**C2 debugging is complete.**

The current C2 should be treated as an **insufficient/failed memory-specific control**, rather than continuing to modify it until it passes.

Next step: **debug C3 using the same step-by-step pipeline.**

In [ ]:
# FINAL C2-v2 CONDITION-LEVEL ANALYSIS
# 1) Average the 4 targets within each distance × filler condition
condition_level = (c2_v2.groupby(["layer", "requested_distance", "filler_idx"])
                   ["delta_log_target_multicontrol"].mean().reset_index(name="condition_delta"))
print("condition counts per layer:"); print(condition_level.groupby("layer").size()); print()

# 2) Summary + 95% CI + one-sample t-test against 0
final_rows = []
for L in sorted(condition_level["layer"].unique()):
    x = condition_level.loc[condition_level["layer"] == L, "condition_delta"].to_numpy()
    n = len(x); mean_log = x.mean(); se = x.std(ddof=1) / np.sqrt(n)
    tcrit = stats.t.ppf(0.975, df=n - 1)
    t_stat, p_value = stats.ttest_1samp(x, popmean=0.0)
    final_rows.append({"layer": L, "n_conditions": n, "mean_log_effect": mean_log,
                       "fold_change": np.exp(mean_log),
                       "ci_low_fold": np.exp(mean_log - tcrit * se),
                       "ci_high_fold": np.exp(mean_log + tcrit * se),
                       "t_stat": t_stat, "p_value": p_value})
final_c2_v2 = pd.DataFrame(final_rows)
print(final_c2_v2.to_string(index=False, formatters={
    "mean_log_effect": "{:.6f}".format, "fold_change": "{:.4f}".format,
    "ci_low_fold": "{:.4f}".format, "ci_high_fold": "{:.4f}".format,
    "t_stat": "{:.4f}".format, "p_value": "{:.6g}".format}))

### C2-v2 Final Result

The multi-control target-only redesign was evaluated at the condition level
(21 distance × filler observations per layer).

- Layer 9: 0.909×, 95% CI [0.843, 0.980], p = 0.015
- Layer 18: 1.045×, 95% CI [0.902, 1.212], p = 0.538
- Layer 27: 0.860×, 95% CI [0.760, 0.974], p = 0.020

The expected positive retention-specific effect is therefore not reliably
supported. Layer 18 remains directionally positive but highly variable, while
layers 9 and 27 show negative effects.

Combined with the five-link debugging trace, this does not indicate an input,
eviction, bookkeeping, or J-Lens-only implementation failure. Rather, the C2
control does not produce the expected memory-specific signal under the current
readout/control design.

C2 debugging is stopped here; no further GPU reruns are justified for this
control.

In [ ]:
for word in ["Paris", "Tokyo", "banana", "lantern"]:
    scores, _ = score_word(word, {"w": encode1(word)}, 1024, 0, layers=[27])
    print(f"{word:8s} layer27 rank = {scores[27]['w'][0]}")

In [ ]:
paris = encode1("Paris")
spec = ai.build_niah_prompt(tok, "Paris", bundle, eviction_distance=1024, in_window=False, filler_idx=0)
ins = tok(spec["prompt"], return_tensors="pt").to(bundle.model.device)
on  = probe.run(ins, nowrite=False, layers=[27], capture_residual=True)   # memory ON
off = probe.run(ins, nowrite=True,  layers=[27], capture_residual=True)   # memory OFF
print("o_t norm WRITE  :", on.o_t(27, pos=-1).norm().item())
print("o_t norm NOWRITE:", off.raw(27, pos=-1).abs().sum().item(), "(should be 0)")
print("resid WRITE  norm:", on.residual(27, pos=-1).norm().item())
print("resid NOWRITE norm:", off.residual(27, pos=-1).norm().item())

In [ ]:
# Original: target itself stored
orig = (df_bias[df_bias["stored_needle"] == df_bias["tested_needle"]]
        [["stored_needle", "layer", "requested_distance", "filler_idx", "p_needle"]]
        .rename(columns={"stored_needle": "target", "p_needle": "p_target_original"}))
# Multi-control baseline: mean log p(target) across the 4 unrelated controls
ctrl = df_c2_multicontrol.copy()
ctrl["log_p_target"] = np.log(ctrl["p_target"] + 1e-30)
ctrl_mean = (ctrl.groupby(["tested_target", "layer", "requested_distance", "filler_idx"])
             .agg(mean_log_p_control=("log_p_target", "mean"),
                  std_log_p_control=("log_p_target", "std"),
                  n_controls=("control_needle", "nunique"))
             .reset_index().rename(columns={"tested_target": "target"}))
# Match original to baseline
c2_v2 = orig.merge(ctrl_mean, on=["target", "layer", "requested_distance", "filler_idx"],
                   validate="one_to_one")
c2_v2["delta_log_target_multicontrol"] = (np.log(c2_v2["p_target_original"] + 1e-30)
                                          - c2_v2["mean_log_p_control"])
summary_c2_v2 = (c2_v2.groupby(["target", "layer"])["delta_log_target_multicontrol"]
                 .agg(["mean", "std", "min", "max", "count"]).reset_index())
print(summary_c2_v2.to_string(index=False))

# C2 five-stage trace — conclusion (2026-09-07)

The signal does not disappear. AHN memory carries the needle: rank 6,703 in the o_t basis and 4,245 memory-isolated (Δ), both far below the ~76,000 chance rank. Suppressing writes (NOWRITE) moves the readout to 62,634 — essentially chance — which confirms the signal comes from compressed memory and not from the backbone prior.

The stage where signal was being lost is the readout basis, not the memory. Reading the full residual stream gives 59,858 (near chance) because the backbone's activity (residual norm ≈ 98) swamps the memory's small contribution (o_t norm ≈ 2.27). The correct basis is o_t, or the WRITE − NOWRITE Δ; the raw residual hides a real signal under backbone noise.

**Caveat.** Absolute probability mass is tiny (p(Paris) = 1e-6) even where the rank is good, as expected for a deep-layer lens readout. Report rank, not probability.

**Scope.** This establishes retention and the correct readout basis for Paris. It does not by itself establish selectivity (Paris over London) or generalisation beyond place names — common-noun needles (e.g. banana) still read near chance at this layer. The full C2 picture is: memory retains content, readable in the o_t basis, and retention is word-dependent.

In [ ]:
# ============================================================
# C2 ABLATION — lantern vs its control (garden)
# Start from the KNOWN positive case: dist=256, layer 18 (+0.47 effect).
# Positive: store lantern.  Control: store garden.
# Both score lantern vs torch. One factor differs: is lantern in memory?
# ============================================================
TARGET, CONTROL, DISTRACT = "lantern", "garden", "torch"
DIST, FILLER = 256, 0          # <-- known positive lantern case (was 1024)

# (3) verify single-token before indexing
lids = tok.encode(f" {TARGET}",   add_special_tokens=False)
tids = tok.encode(f" {DISTRACT}", add_special_tokens=False)
assert len(lids) == 1, (TARGET, lids)
assert len(tids) == 1, (DISTRACT, tids)
lid, tid = lids[0], tids[0]

def trace(stored):
    spec = ai.build_niah_prompt(tok, stored, bundle,
        eviction_distance=DIST, in_window=False, filler_idx=FILLER)
    ins = tok(spec["prompt"], return_tensors="pt").to(bundle.model.device)
    # (4) match the C2 sweep exactly — no residual capture
    cap = probe.run(ins, nowrite=False, layers=EXP["layers"], capture_residual=False)
    out = {}
    for L in EXP["layers"]:
        lg = ai.readout_logits(cap.o_t(L, pos=-1), bundle, lens=lens, layer=L)
        pn = ai.token_prob(lg, lid); pd = ai.token_prob(lg, tid)
        out[L] = dict(
            rank_lantern = ai.token_rank(lg, lid),
            rank_torch   = ai.token_rank(lg, tid),
            logratio     = np.log(pn + 1e-30) - np.log(pd + 1e-30),
        )
    return spec, ins["input_ids"][0], out

spec_pos, ids_pos, pos = trace(TARGET)
spec_ctl, ids_ctl, ctl = trace(CONTROL)

# (1) structural match check
print("MATCH CHECK (identical except stored word):")
print(f"  n_tokens       pos={spec_pos['n_tokens']}   ctl={spec_ctl['n_tokens']}")
print(f"  needle_pos     pos={spec_pos['needle_pos']}   ctl={spec_ctl['needle_pos']}")
print(f"  evicted        pos={spec_pos['needle_is_evicted']}   ctl={spec_ctl['needle_is_evicted']}")
print(f"  evict_distance pos={spec_pos['actual_eviction_distance']}   ctl={spec_ctl['actual_eviction_distance']}")

# (2) token-by-token: prove they differ at exactly one position
if len(ids_pos) == len(ids_ctl):
    diffs = (ids_pos != ids_ctl).nonzero().flatten().tolist()
    print(f"\nTOKEN DIFF: {len(diffs)} position(s) differ", 
          f"-> {diffs}" if len(diffs) <= 5 else f"(first 5: {diffs[:5]})")
    for d in diffs[:5]:
        print(f"    pos {d}: pos={tok.decode([ids_pos[d]])!r}  ctl={tok.decode([ids_ctl[d]])!r}")
else:
    print(f"\nTOKEN DIFF: LENGTH MISMATCH {len(ids_pos)} vs {len(ids_ctl)} — not one-factor!")

print(f"\nchance rank ~76,000 | logratio = log[p(lantern)/p(torch)]")
print(f"{'layer':>5} | {'POSITIVE (lantern)':^32} | {'CONTROL (garden)':^32}")
print(f"{'':>5} | {'rank_lan':>9} {'rank_tor':>9} {'logratio':>10} | {'rank_lan':>9} {'rank_tor':>9} {'logratio':>10}")
print("-"*76)
for L in EXP["layers"]:
    p, c = pos[L], ctl[L]
    print(f"{L:>5} | {p['rank_lantern']:>9,} {p['rank_torch']:>9,} {p['logratio']:>+10.3f} "
          f"| {c['rank_lantern']:>9,} {c['rank_torch']:>9,} {c['logratio']:>+10.3f}")

print(f"\nC2 effect (positive − control logratio):")
for L in EXP["layers"]:
    print(f"  layer {L}: {pos[L]['logratio'] - ctl[L]['logratio']:+.3f}")

In [ ]:
# ============================================================
# C2 ABLATION — Paris vs its control (river)   [dist=256, Paris's positive case]
# Positive: store Paris.  Control: store river.  Both score Paris vs London.
# Same distance as the lantern run -> direct contrast.
# ============================================================
TARGET, CONTROL, DISTRACT = "Paris", "river", "London"
DIST, FILLER = 256, 0

lids = tok.encode(f" {TARGET}",   add_special_tokens=False)
tids = tok.encode(f" {DISTRACT}", add_special_tokens=False)
assert len(lids) == 1, (TARGET, lids)
assert len(tids) == 1, (DISTRACT, tids)
lid, tid = lids[0], tids[0]

def trace(stored):
    spec = ai.build_niah_prompt(tok, stored, bundle,
        eviction_distance=DIST, in_window=False, filler_idx=FILLER)
    ins = tok(spec["prompt"], return_tensors="pt").to(bundle.model.device)
    cap = probe.run(ins, nowrite=False, layers=EXP["layers"], capture_residual=False)
    out = {}
    for L in EXP["layers"]:
        lg = ai.readout_logits(cap.o_t(L, pos=-1), bundle, lens=lens, layer=L)
        pn = ai.token_prob(lg, lid); pd = ai.token_prob(lg, tid)
        out[L] = dict(
            rank_target   = ai.token_rank(lg, lid),
            rank_distract = ai.token_rank(lg, tid),
            logratio      = np.log(pn + 1e-30) - np.log(pd + 1e-30),
        )
    return spec, ins["input_ids"][0], out

spec_pos, ids_pos, pos = trace(TARGET)
spec_ctl, ids_ctl, ctl = trace(CONTROL)

print("MATCH CHECK (identical except stored word):")
print(f"  n_tokens       pos={spec_pos['n_tokens']}   ctl={spec_ctl['n_tokens']}")
print(f"  needle_pos     pos={spec_pos['needle_pos']}   ctl={spec_ctl['needle_pos']}")
print(f"  evicted        pos={spec_pos['needle_is_evicted']}   ctl={spec_ctl['needle_is_evicted']}")
print(f"  evict_distance pos={spec_pos['actual_eviction_distance']}   ctl={spec_ctl['actual_eviction_distance']}")

if len(ids_pos) == len(ids_ctl):
    diffs = (ids_pos != ids_ctl).nonzero().flatten().tolist()
    print(f"\nTOKEN DIFF: {len(diffs)} position(s) differ -> {diffs}")
    for d in diffs[:5]:
        print(f"    pos {d}: pos={tok.decode([ids_pos[d]])!r}  ctl={tok.decode([ids_ctl[d]])!r}")
else:
    print(f"\nTOKEN DIFF: LENGTH MISMATCH {len(ids_pos)} vs {len(ids_ctl)} — not one-factor!")

print(f"\nchance rank ~76,000 | logratio = log[p(Paris)/p(London)]")
print(f"{'layer':>5} | {'POSITIVE (Paris)':^34} | {'CONTROL (river)':^34}")
print(f"{'':>5} | {'rank_Paris':>10} {'rank_London':>11} {'logratio':>10} | {'rank_Paris':>10} {'rank_London':>11} {'logratio':>10}")
print("-"*82)
for L in EXP["layers"]:
    p, c = pos[L], ctl[L]
    print(f"{L:>5} | {p['rank_target']:>10,} {p['rank_distract']:>11,} {p['logratio']:>+10.3f} "
          f"| {c['rank_target']:>10,} {c['rank_distract']:>11,} {c['logratio']:>+10.3f}")

print(f"\nC2 effect (positive − control logratio):")
for L in EXP["layers"]:
    print(f"  layer {L}: {pos[L]['logratio'] - ctl[L]['logratio']:+.3f}")

In [ ]:
pid = tok.encode(" Paris",  add_special_tokens=False)[0]
did = tok.encode(" London", add_special_tokens=False)[0]

def c2_effect(dist, L=27):
    def lr(stored):
        spec = ai.build_niah_prompt(tok, stored, bundle, eviction_distance=dist, in_window=False, filler_idx=0)
        ins = tok(spec["prompt"], return_tensors="pt").to(bundle.model.device)
        cap = probe.run(ins, nowrite=False, layers=[L], capture_residual=False)
        lg = ai.readout_logits(cap.o_t(L, pos=-1), bundle, lens=lens, layer=L)
        return np.log(ai.token_prob(lg,pid)+1e-30) - np.log(ai.token_prob(lg,did)+1e-30)
    return lr("Paris") - lr("river")

for d in [64, 256, 1024]:
    print(f"dist {d:>5}: Paris C2 effect (L27) = {c2_effect(d):+.3f}")

In [ ]:
# ============================================================
# C2 ABLATION — firm-up: Paris vs lantern across ALL distances at layer 27
# Same one-factor design (target stored vs control stored, score target/distractor).
# ============================================================
L = 27
DISTANCES = EXP["eviction_distances"]     # [64, 256, 512, 1024, 2048, 4096, 8192]
FILLERS   = range(EXP["n_filler_variants"])  # 0,1,2

CASES = {
    "Paris":   dict(control="river",  distractor="London"),
    "lantern": dict(control="garden", distractor="torch"),
}

def logratio(stored, target_id, distr_id, dist, filler):
    spec = ai.build_niah_prompt(tok, stored, bundle,
        eviction_distance=dist, in_window=False, filler_idx=filler)
    ins = tok(spec["prompt"], return_tensors="pt").to(bundle.model.device)
    cap = probe.run(ins, nowrite=False, layers=[L], capture_residual=False)
    lg  = ai.readout_logits(cap.o_t(L, pos=-1), bundle, lens=lens, layer=L)
    return (np.log(ai.token_prob(lg, target_id)+1e-30)
            - np.log(ai.token_prob(lg, distr_id)+1e-30))

for target, cfg in CASES.items():
    tid = tok.encode(f" {target}",            add_special_tokens=False)[0]
    did = tok.encode(f" {cfg['distractor']}", add_special_tokens=False)[0]
    effects = []
    print(f"\n=== {target} (control={cfg['control']}, distractor={cfg['distractor']}) — layer {L} ===")
    print(f"{'dist':>6} {'filler':>7} | {'C2 effect':>10}")
    print("-"*30)
    for dist in DISTANCES:
        for f in FILLERS:
            pos = logratio(target,           tid, did, dist, f)
            ctl = logratio(cfg["control"],   tid, did, dist, f)
            eff = pos - ctl
            effects.append(eff)
            print(f"{dist:>6} {f:>7} | {eff:>+10.3f}")
    arr = np.array(effects)
    print(f"  mean={arr.mean():+.3f}  median={np.median(arr):+.3f}  "
          f"frac>0={ (arr>0).mean():.2f}  n={len(arr)}")